H-1/H-2 are the scripts in scripts/. The H-3 to H-5 cells here are the first draft. The team version is sentinelai/data/ + scripts/build_features.py (uses the ALL-1 contract column names).

In [ ]:
!pip install pandas requests

In [ ]:
# Step 0: download the AIT Netflow archives + label_info.txt (needed by H-1 and H-5). Skips files already here.
# Official sizes + MD5s come from the Zenodo API (record 13168643); any mismatch stops the cell.
import hashlib, requests
from pathlib import Path
DL = Path('sentinelai_ait_data/downloads'); DL.mkdir(parents=True, exist_ok=True)
rec = requests.get('https://zenodo.org/api/records/13168643', timeout=60).json()
files = {f['key']: f for f in rec['files']}
want = ['label_info.txt'] + [f'{n}_netflows.zip' for n in
        ['fox', 'harrison', 'russellmitchell', 'santos', 'shaw', 'wardbeck', 'wheeler', 'wilson']]

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()

for name in want:
    f, p = files[name], DL / name
    if not p.exists():
        with requests.get(f['links']['self'], stream=True, timeout=(30, 300)) as r:
            r.raise_for_status()
            with open(p, 'wb') as out:
                for b in r.iter_content(1 << 20):
                    out.write(b)
    ok = md5(p) == f['checksum'].split(':', 1)[1] and p.stat().st_size == f['size']
    print(f'{name:<30} {p.stat().st_size:>11,} bytes   md5+size match official: {ok}')
    assert ok, f'{name} does not match the official Zenodo checksum - stop'

In [ ]:
# H-1 Step 2: read the OFFICIAL checksums from the Zenodo REST API (publisher metadata)
# and compare them character-by-character with (a) the script's hardcoded values and (b) the files on disk.
import requests, hashlib, json, re
from pathlib import Path

DL = Path('sentinelai_ait_data/downloads')
official = {}
for rec in ['13168643', '6610489']:          # v2 (used by the script) and the original v1 record
    r = requests.get(f'https://zenodo.org/api/records/{rec}', timeout=60)
    print(rec, 'HTTP', r.status_code)
    if r.ok:
        d = r.json()
        print('  title:', d['metadata'].get('title'), '| version:', d['metadata'].get('version'),
              '| published:', d['metadata'].get('publication_date'), '| doi:', d.get('doi'))
        for f in d.get('files', []):
            print(f"  {f['key']:<32} {f['size']:>11,}  {f['checksum']}")
            if rec == '13168643':
                official[f['key']] = (f['checksum'].split(':',1)[1], f['size'])

# Hardcoded values from the script (parsed from source, not retyped)
src = open('sentinelai_ait_verify.py').read()
script_md5 = dict(re.findall(r'"([a-z]+_netflows\.zip)":\s*"([0-9a-f]{32})"', src))

def md5_file(p):
    h = hashlib.md5()
    with open(p,'rb') as f:
        for b in iter(lambda: f.read(1<<20), b''): h.update(b)
    return h.hexdigest()

print('\nfile | official(zenodo v2) | script | actual | official==actual | script==official | size official/actual')
for name in sorted(official):
    p = DL/name
    act = md5_file(p) if p.exists() else None
    off, size = official[name]
    sc = script_md5.get(name)
    print(name, off, sc, act, off==act, sc==off, size, p.stat().st_size if p.exists() else None)
    if sc and sc != off:
        diff = [i+1 for i,(a,b) in enumerate(zip(sc,off)) if a!=b]
        print('   -> script differs from official at character positions', diff, '| script:', sc[8:13], 'official:', off[8:13])

In [ ]:
# H-1 Step 3: read label_info.txt verbatim (repr shows hidden whitespace) and peek at each CSV header inside the ZIPs
import zipfile, io
from pathlib import Path
DL = Path('sentinelai_ait_data/downloads')
raw = (DL/'label_info.txt').read_bytes()
print('label_info.txt bytes:', len(raw), '| BOM:', raw[:3]==b'\xef\xbb\xbf', '| CRLF:', b'\r\n' in raw)
for i, line in enumerate(raw.decode('utf-8-sig').splitlines(), 1):
    print(f'{i:3d}: {line!r}')
print()
with zipfile.ZipFile(DL/'fox_netflows.zip') as z:
    for info in z.infolist():
        with z.open(info) as f:
            head = [f.readline().decode('utf-8','replace').rstrip('\n') for _ in range(3)]
        print(info.filename, info.file_size, info.compress_type)
        for h in head: print('   ', h[:600])

In [ ]:
# H-1 Step 4: read ONLY the header line of every CSV in every archive (streamed from the ZIP, nothing extracted)
# and group files by identical raw header, to check schema consistency before any counting.
import zipfile, hashlib
from pathlib import Path
from collections import defaultdict
DL = Path('sentinelai_ait_data/downloads')
NETS = ['fox','harrison','russellmitchell','santos','shaw','wardbeck','wheeler','wilson']
groups = defaultdict(list)
for n in NETS:
    with zipfile.ZipFile(DL/f'{n}_netflows.zip') as z:
        for info in z.infolist():
            with z.open(info) as f:
                hdr = f.readline().decode('utf-8').rstrip('\r\n')
                first = f.readline().decode('utf-8','replace').rstrip('\r\n')
            groups[hdr].append((n, info.filename, first.count(',')+1))
for k,(hdr, members) in enumerate(groups.items()):
    cols = hdr.split(',')
    print(f'=== SCHEMA {k}: {len(cols)} header fields; used by {len(members)} files')
    print('   files:', [(m[0], m[1]) for m in members])
    print('   field count in first data row:', sorted(set(m[2] for m in members)))
    print('   RAW HEADER:', hdr)
    print()

In [ ]:
%%writefile h1_ait_verify.py
#!/usr/bin/env python3
"""
SentinelAI H-1 - AIT Netflow Data Set verification (v2, chunked).

What it does (read-only with respect to the dataset):
  1. Reads official file sizes + MD5s from the Zenodo REST API (record 13168643).
  2. Verifies every downloaded archive: byte size, MD5, ZIP CRC test (testzip).
  3. Parses label_info.txt into an exact, protocol-aware benign(0)/malicious(1) map.
  4. Streams every CSV straight out of its ZIP in chunks (no extraction, the
     original archives are never modified) and records: rows, columns, exact
     column names, label column, protocol, missing values, duplicates, parsing
     problems and exact label value counts.
  5. Writes the H-1 evidence files into --out-dir.

Unknown labels are NEVER treated as benign: anything that is not an exact
match to a label listed for that protocol in label_info.txt is "unmapped".

Run:  python h1_ait_verify.py --data-dir ./sentinelai_ait_data --out-dir ./h1_outputs
"""
from __future__ import annotations

import argparse
import datetime as dt
import hashlib
import io
import json
import platform
import re
import sys
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import requests

# --------------------------------------------------------------------------
# Constants
# --------------------------------------------------------------------------
ZENODO_RECORD_ID = "13168643"                      # AIT Netflow Data Set v2 (2024-08-02)
ZENODO_DOI = "10.5281/zenodo.13168643"
ZENODO_API = f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}"
NETWORKS = ["fox", "harrison", "russellmitchell", "santos",
            "shaw", "wardbeck", "wheeler", "wilson"]

# MD5s hardcoded in the ORIGINAL sentinelai_ait_verify.py, kept only so the
# report can show which of them were transcribed correctly.
SCRIPT_V1_MD5 = {
    "fox_netflows.zip": "d3024ef6033fbd45982464f6b6f50984",
    "harrison_netflows.zip": "840224491fb31d548e7ee0ecbd951db5",
    "russellmitchell_netflows.zip": "290aa5b8a0b9d0515b7e99329c467364",  # typo
    "santos_netflows.zip": "e217db1c0307a8966db9e79609789fcb",
    "shaw_netflows.zip": "25c5a62572d37f088353cf7bbd58f590",
    "wardbeck_netflows.zip": "3a73afcaef00a1bbe53f1c39060c2f17",
    "wheeler_netflows.zip": "2c45193c0babc8ec45f356f3e39118a6",
    "wilson_netflows.zip": "2c10183cf7aca03e9fb80408a360f34c",
}

CHUNK_ROWS = 200_000                                # rows per pandas chunk (RAM bound)
NA_TOKENS = ["NA", "N/A", "nan", "NaN", "null", "NULL", "None", "-"]  # reported, not converted
LABEL_COLUMN = "label"
# Columns that only exist in TCP Tstat logs; used to cross-check the protocol
# inferred from the file name against the actual schema.
TCP_ONLY_MARKERS = ("c_syn_cnt", "c_fin_cnt", "c_rst_cnt")


# --------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------
def md5_file(path: Path) -> str:
    """MD5 of a file, read in 1 MiB blocks."""
    h = hashlib.md5()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def clean_column(raw: str) -> str:
    """Tstat headers look like '#15#c_ip:1' or 'c_port:2'.
    Return the bare field name ('c_ip', 'c_port') for readability.
    The raw names are ALWAYS reported as well."""
    name = re.sub(r"^#(\d+#)?", "", raw)
    name = re.sub(r":\d+$", "", name)
    return name


class LineCountingReader(io.RawIOBase):
    """Wraps a binary stream and counts newline bytes as pandas reads it, so we
    can reconcile pandas' row count against the physical line count in ONE pass."""

    def __init__(self, raw):
        self.raw = raw
        self.newlines = 0
        self.nbytes = 0
        self.last_byte = b""

    def readable(self):
        return True

    def readinto(self, b):
        data = self.raw.read(len(b))
        n = len(data)
        if n:
            b[:n] = data
            self.newlines += data.count(b"\n")
            self.nbytes += n
            self.last_byte = data[-1:]
        return n

    def physical_lines(self) -> int:
        if self.nbytes == 0:
            return 0
        return self.newlines + (0 if self.last_byte == b"\n" else 1)


def fetch_official_metadata():
    """Return {filename: {'md5':..., 'bytes':...}} from Zenodo, plus record info."""
    r = requests.get(ZENODO_API, timeout=60)
    r.raise_for_status()
    d = r.json()
    files = {}
    for f in d.get("files", []):
        algo, _, value = f["checksum"].partition(":")
        if algo != "md5":
            raise RuntimeError(f"Unexpected checksum algorithm {algo} for {f['key']}")
        files[f["key"]] = {"md5": value.lower(), "bytes": int(f["size"])}
    info = {"record_id": ZENODO_RECORD_ID, "doi": d.get("doi"),
            "title": d["metadata"].get("title"),
            "publication_date": d["metadata"].get("publication_date"),
            "api_url": ZENODO_API,
            "retrieved_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")}
    return files, info


def parse_label_info(path: Path):
    """Parse the four sections 'Benign/Malicious labels in TCP/UDP:'.
    Exact strings are kept (only the line terminator is removed).
    Returns {protocol: {label: 0|1}} and a list of problems found in the file."""
    text = path.read_bytes().decode("utf-8-sig")
    mapping: dict[str, dict[str, int]] = {"TCP": {}, "UDP": {}}
    problems = []
    current = None
    seen_sections = []
    for lineno, line in enumerate(text.splitlines(), 1):
        if line.strip() == "":
            continue
        m = re.fullmatch(r"(Benign|Malicious) labels in (TCP|UDP):", line.strip())
        if m:
            current = (m.group(2), 0 if m.group(1) == "Benign" else 1)
            seen_sections.append(line.strip())
            continue
        if current is None:
            problems.append(f"line {lineno}: text before first section: {line!r}")
            continue
        if line != line.strip():
            problems.append(f"line {lineno}: label has leading/trailing whitespace: {line!r}")
        proto, y = current
        if line in mapping[proto] and mapping[proto][line] != y:
            problems.append(f"line {lineno}: {line!r} listed as both benign and malicious in {proto}")
        mapping[proto][line] = y
    if len(seen_sections) != 4:
        problems.append(f"expected 4 sections, found {len(seen_sections)}: {seen_sections}")
    return mapping, problems


# --------------------------------------------------------------------------
# Per-CSV inspection (streamed, chunked)
# --------------------------------------------------------------------------
def inspect_csv(zf: zipfile.ZipFile, member: str, protocol: str, label_map: dict):
    """Stream one CSV out of the ZIP and return (file_record, label_counter)."""
    rec = {"parse_error": ""}
    label_counts: Counter = Counter()
    row_hashes, feat_hashes, ybin = [], [], []
    missing = None
    na_tok = None
    rows = 0
    ts_min, ts_max = np.inf, -np.inf
    sample_types = {}

    with zf.open(member) as raw:
        counter = LineCountingReader(raw)
        handle = io.BufferedReader(counter, buffer_size=1 << 20)
        try:
            reader = pd.read_csv(handle, dtype=str, keep_default_na=False, na_filter=False,
                                 chunksize=CHUNK_ROWS, on_bad_lines="error",
                                 encoding="utf-8", skip_blank_lines=False)
            for i, chunk in enumerate(reader):
                if i == 0:
                    raw_cols = [str(c) for c in chunk.columns]
                    rec["columns_raw"] = raw_cols
                    rec["columns_clean"] = [clean_column(c) for c in raw_cols]
                    if len(set(raw_cols)) != len(raw_cols):
                        rec["parse_error"] += "duplicate column names; "
                    # Type profile on the first chunk only (cheap; labelled as a sample).
                    for c in raw_cols:
                        s = chunk[c]
                        nonempty = s[s != ""]
                        num = pd.to_numeric(nonempty, errors="coerce")
                        n_bad = int(num.isna().sum())
                        sample_types[clean_column(c)] = (
                            "empty" if len(nonempty) == 0 else
                            "numeric" if n_bad == 0 else
                            "text" if n_bad == len(nonempty) else
                            f"mixed({n_bad} non-numeric of {len(nonempty)})")
                    missing = pd.Series(0, index=raw_cols, dtype="int64")
                    na_tok = pd.Series(0, index=raw_cols, dtype="int64")
                    ts_col = next((c for c in raw_cols if clean_column(c) in ("first", "c_first_abs")), None)
                    rec["time_column"] = clean_column(ts_col) if ts_col else ""

                rows += len(chunk)
                missing += (chunk == "").sum()                     # truly empty cells
                na_tok += chunk.isin(NA_TOKENS).sum()              # textual NA-like tokens

                # Exact label value counts (no stripping, no case folding).
                labels = chunk[LABEL_COLUMN]
                label_counts.update(labels.value_counts().to_dict())

                # Row hashes for duplicate detection across the whole file.
                row_hashes.append(pd.util.hash_pandas_object(chunk, index=False).to_numpy())
                feat_hashes.append(pd.util.hash_pandas_object(
                    chunk.drop(columns=[LABEL_COLUMN]), index=False).to_numpy())
                ybin.append(labels.map(label_map).fillna(-1).to_numpy(dtype=np.int8))

                if ts_col:
                    t = pd.to_numeric(chunk[ts_col], errors="coerce")
                    if t.notna().any():
                        ts_min = min(ts_min, float(t.min()))
                        ts_max = max(ts_max, float(t.max()))
        except Exception as exc:                                   # record, never hide
            rec["parse_error"] += f"{type(exc).__name__}: {exc}"

        # Drain anything pandas did not consume so the line count is complete.
        while handle.read(1 << 20):
            pass

    rec["rows"] = rows
    rec["physical_lines_incl_header"] = counter.physical_lines()
    rec["uncompressed_bytes_read"] = counter.nbytes
    rec["row_count_reconciles"] = (counter.physical_lines() - 1 == rows)
    rec["missing"] = missing
    rec["na_tokens"] = na_tok
    rec["sample_types"] = sample_types
    rec["time_min"] = ts_min if np.isfinite(ts_min) else None
    rec["time_max"] = ts_max if np.isfinite(ts_max) else None

    if rows:
        rh = np.concatenate(row_hashes)
        fh = np.concatenate(feat_hashes)
        yb = np.concatenate(ybin)
        rec["exact_duplicate_rows"] = int(rows - np.unique(rh).size)
        # Identical feature values (all columns except label) but different binary label.
        g = pd.DataFrame({"f": fh, "y": yb}).groupby("f")["y"].agg(["nunique", "size"])
        conflict = g[g["nunique"] > 1]
        rec["conflicting_label_groups"] = int(len(conflict))
        rec["rows_in_conflicting_label_groups"] = int(conflict["size"].sum())
    else:
        rec["exact_duplicate_rows"] = 0
        rec["conflicting_label_groups"] = 0
        rec["rows_in_conflicting_label_groups"] = 0
    return rec, label_counts


# --------------------------------------------------------------------------
# Main
# --------------------------------------------------------------------------
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", default="./sentinelai_ait_data")
    ap.add_argument("--out-dir", default="./h1_outputs")
    ap.add_argument("--offline-test-metadata", default=None,
                    help="TESTING ONLY: JSON file replacing the Zenodo API call")
    args = ap.parse_args()
    downloads = Path(args.data_dir).resolve() / "downloads"
    out = Path(args.out_dir).resolve()
    out.mkdir(parents=True, exist_ok=True)

    # ---- 1. Official metadata ------------------------------------------------
    if args.offline_test_metadata:                    # used only for local unit testing
        official = json.loads(Path(args.offline_test_metadata).read_text())
        record_info = {"record_id": "TEST", "doi": "TEST", "title": "TEST", "publication_date": "",
                       "api_url": "", "retrieved_utc": ""}
    else:
        official, record_info = fetch_official_metadata()
    print(f"[zenodo] {record_info['title']} | doi {record_info['doi']} | {len(official)} files listed")

    # ---- 2. Archive verification (all archives checked; no early abort) -----
    arch_rows = []
    verified = {}
    for fname in ["label_info.txt"] + [f"{n}_netflows.zip" for n in NETWORKS]:
        p = downloads / fname
        off = official.get(fname, {})
        row = {"file": fname,
               "network": fname.replace("_netflows.zip", "") if fname.endswith(".zip") else "",
               "official_md5": off.get("md5", ""), "official_bytes": off.get("bytes", ""),
               "exists": p.exists()}
        if p.exists():
            row["actual_md5"] = md5_file(p)
            row["actual_bytes"] = p.stat().st_size
        else:
            row["actual_md5"], row["actual_bytes"] = "", ""
        row["md5_match"] = bool(off) and row["actual_md5"] == off.get("md5")
        row["size_match"] = bool(off) and row["actual_bytes"] == off.get("bytes")
        row["zip_testzip"] = ""
        row["zip_members"] = ""
        zip_ok = True
        if fname.endswith(".zip") and p.exists():
            try:
                with zipfile.ZipFile(p) as zf:
                    bad = zf.testzip()                     # reads every member, checks CRC-32
                    row["zip_testzip"] = "OK (all CRCs valid)" if bad is None else f"BAD member: {bad}"
                    zip_ok = bad is None
                    row["zip_members"] = json.dumps(
                        [{"name": i.filename, "uncompressed_bytes": i.file_size} for i in zf.infolist()])
            except zipfile.BadZipFile as exc:
                row["zip_testzip"] = f"BadZipFile: {exc}"
                zip_ok = False
        elif fname.endswith(".zip"):
            zip_ok = False
        row["script_v1_expected_md5"] = SCRIPT_V1_MD5.get(fname, "")
        row["script_v1_md5_equals_official"] = (SCRIPT_V1_MD5.get(fname) == off.get("md5")) if fname in SCRIPT_V1_MD5 else ""
        row["verified"] = bool(row["md5_match"] and row["size_match"] and zip_ok)
        verified[fname] = row["verified"]
        arch_rows.append(row)
        print(f"[archive] {fname:<30} md5_match={row['md5_match']} size_match={row['size_match']} "
              f"zip={row['zip_testzip'] or 'n/a'} -> verified={row['verified']}")
    pd.DataFrame(arch_rows).to_csv(out / "archive_checksums.csv", index=False)

    if not verified["label_info.txt"]:
        sys.exit("label_info.txt failed verification - stopping before label mapping.")

    # ---- 3. Official label mapping -------------------------------------------
    label_map, label_problems = parse_label_info(downloads / "label_info.txt")
    for proto in ("TCP", "UDP"):
        b = sum(1 for v in label_map[proto].values() if v == 0)
        m = sum(1 for v in label_map[proto].values() if v == 1)
        print(f"[labels] {proto}: {b} benign, {m} malicious labels in label_info.txt")
    for pr in label_problems:
        print("[labels][problem]", pr)

    # ---- 4. CSV inspection ---------------------------------------------------
    file_rows, label_rows = [], []
    schema_ids: dict[tuple, str] = {}
    observed = {"TCP": Counter(), "UDP": Counter()}
    for net in NETWORKS:
        zname = f"{net}_netflows.zip"
        if not verified.get(zname):
            print(f"[skip] {zname} not verified - its CSVs are NOT inspected or counted")
            continue
        with zipfile.ZipFile(downloads / zname) as zf:
            members = [i.filename for i in zf.infolist() if i.filename.lower().endswith(".csv")]
            for member in sorted(members):
                base = Path(member).name.lower()
                protocol = "UDP" if base.startswith("udp") else "TCP" if base.startswith("tcp") else "UNKNOWN"
                completeness = "nocomplete" if "nocomplete" in base else "complete"
                pmap = label_map.get(protocol, {})
                rec, counts = inspect_csv(zf, member, protocol, pmap)
                cols = rec.get("columns_raw", [])
                clean = rec.get("columns_clean", [])
                sid = schema_ids.setdefault(tuple(cols), f"S{len(schema_ids) + 1}")

                # Protocol cross-check: TCP files must have TCP-only fields, UDP files must not.
                has_tcp_fields = all(m in clean for m in TCP_ONLY_MARKERS)
                proto_check = ("consistent" if (protocol == "TCP") == has_tcp_fields
                               else "INCONSISTENT with schema")

                normal = suspicious = unmapped = 0
                strip_variants = 0
                for value, n in sorted(counts.items(), key=lambda kv: -kv[1]):
                    y = pmap.get(value)
                    if y is None:
                        other = "UDP" if protocol == "TCP" else "TCP"
                        if value in label_map.get(other, {}):
                            cat = f"not listed for {protocol} (listed under {other})"
                        elif value.strip() in pmap:
                            cat = "unmapped: matches an official label only after whitespace strip"
                            strip_variants += n
                        else:
                            cat = "unmapped: not in label_info.txt"
                        unmapped += n
                    elif y == 0:
                        cat = "benign (label_info.txt)"; normal += n
                    else:
                        cat = "malicious (label_info.txt)"; suspicious += n
                    observed[protocol][value] += n
                    label_rows.append({"network": net, "file": Path(member).name, "protocol": protocol,
                                       "original_label": value, "official_category": cat,
                                       "binary_label": "" if y is None else y, "flow_count": n})

                miss = rec["missing"] if rec.get("missing") is not None else pd.Series(dtype="int64")
                natk = rec["na_tokens"] if rec.get("na_tokens") is not None else pd.Series(dtype="int64")
                file_rows.append({
                    "network": net, "file": Path(member).name, "protocol": protocol,
                    "flow_completeness": completeness,
                    "protocol_vs_schema_check": proto_check,
                    "rows": rec["rows"],
                    "physical_lines_incl_header": rec["physical_lines_incl_header"],
                    "row_count_reconciles": rec["row_count_reconciles"],
                    "n_columns": len(cols), "schema_id": sid,
                    "label_column": LABEL_COLUMN if LABEL_COLUMN in cols else "NOT FOUND",
                    "columns_raw_json": json.dumps(cols),
                    "columns_clean_json": json.dumps(clean),
                    "empty_cells_total": int(miss.sum()),
                    "columns_with_empty_cells": int((miss > 0).sum()),
                    "empty_cells_by_column_json": json.dumps({clean_column(k): int(v) for k, v in miss.items() if v}),
                    "na_token_cells_total": int(natk.sum()),
                    "na_tokens_by_column_json": json.dumps({clean_column(k): int(v) for k, v in natk.items() if v}),
                    "exact_duplicate_rows": rec["exact_duplicate_rows"],
                    "conflicting_label_groups": rec["conflicting_label_groups"],
                    "rows_in_conflicting_label_groups": rec["rows_in_conflicting_label_groups"],
                    "distinct_labels": len(counts),
                    "normal_flows": normal, "suspicious_flows": suspicious, "unmapped_flows": unmapped,
                    "labels_matching_only_after_strip": strip_variants,
                    "label_counts_sum_equals_rows": (normal + suspicious + unmapped) == rec["rows"],
                    "time_column": rec.get("time_column", ""),
                    "time_min": rec["time_min"], "time_max": rec["time_max"],
                    "mixed_or_text_columns_first_chunk_json": json.dumps(
                        {k: v for k, v in rec["sample_types"].items() if v != "numeric"}),
                    "uncompressed_bytes_read": rec["uncompressed_bytes_read"],
                    "parse_error": rec["parse_error"],
                })
                print(f"[csv] {net}/{Path(member).name}: rows={rec['rows']:,} cols={len(cols)} "
                      f"schema={sid} normal={normal:,} suspicious={suspicious:,} unmapped={unmapped:,} "
                      f"dups={rec['exact_duplicate_rows']:,} reconcile={rec['row_count_reconciles']} "
                      f"err={rec['parse_error'] or '-'}")

    fq = pd.DataFrame(file_rows)
    fq.to_csv(out / "csv_file_quality_report.csv", index=False)
    pd.DataFrame(label_rows).to_csv(out / "label_counts_by_file.csv", index=False)

    # ---- 5. Per-network attack counts ----------------------------------------
    net_rows = []
    for net in NETWORKS:
        d = fq[fq["network"] == net] if len(fq) else fq
        row = {"network": net, "archive_verified": verified.get(f"{net}_netflows.zip", False),
               "csv_files": len(d)}
        for key, sub in (("all", d),
                         ("tcp_complete", d[(d.protocol == "TCP") & (d.flow_completeness == "complete")]),
                         ("tcp_nocomplete", d[(d.protocol == "TCP") & (d.flow_completeness == "nocomplete")]),
                         ("tcp", d[d.protocol == "TCP"]), ("udp", d[d.protocol == "UDP"])):
            pre = "" if key == "all" else f"{key}_"
            row[f"{pre}total_flows"] = int(sub["rows"].sum())
            row[f"{pre}normal_flows"] = int(sub["normal_flows"].sum())
            row[f"{pre}suspicious_flows"] = int(sub["suspicious_flows"].sum())
            row[f"{pre}unmapped_flows"] = int(sub["unmapped_flows"].sum())
        row["suspicious_pct"] = round(100 * row["suspicious_flows"] / row["total_flows"], 4) if row["total_flows"] else None
        row["counts_reconcile"] = (row["normal_flows"] + row["suspicious_flows"] + row["unmapped_flows"]
                                   == row["total_flows"]) and len(d) > 0 and bool(d["row_count_reconciles"].all())
        net_rows.append(row)
    nt = pd.DataFrame(net_rows)
    total = {"network": "TOTAL", "archive_verified": bool(nt["archive_verified"].all()),
             "csv_files": int(nt["csv_files"].sum())}
    for c in nt.columns:
        if c.endswith("_flows"):
            total[c] = int(nt[c].sum())
    total["suspicious_pct"] = round(100 * total["suspicious_flows"] / total["total_flows"], 4) if total["total_flows"] else None
    total["counts_reconcile"] = bool(nt["counts_reconcile"].all())
    nt = pd.concat([nt, pd.DataFrame([total])], ignore_index=True)
    nt.to_csv(out / "ait_network_attack_counts.csv", index=False)

    # ---- 6. label_mapping.json -------------------------------------------------
    mapping_doc = {
        "source_file": "label_info.txt",
        "source_md5": next(r["actual_md5"] for r in arch_rows if r["file"] == "label_info.txt"),
        "source_record": record_info,
        "encoding": {"Normal": 0, "Suspicious": 1},
        "rules": [
            "Mapping is protocol-aware: TCP CSVs use the TCP sections, UDP CSVs use the UDP sections.",
            "Matching is exact (case- and whitespace-sensitive) against label_info.txt lines.",
            "Benign labels -> 0 (Normal); malicious labels -> 1 (Suspicious).",
            "Any other value is UNMAPPED and is never treated as benign.",
        ],
        "label_info_parse_problems": label_problems,
        "TCP": {"benign_0": sorted(k for k, v in label_map["TCP"].items() if v == 0),
                "malicious_1": sorted(k for k, v in label_map["TCP"].items() if v == 1)},
        "UDP": {"benign_0": sorted(k for k, v in label_map["UDP"].items() if v == 0),
                "malicious_1": sorted(k for k, v in label_map["UDP"].items() if v == 1)},
        "observed_in_csvs": {
            p: {"unmapped_labels": {k: v for k, v in observed[p].items() if k not in label_map[p]},
                "official_labels_never_observed": sorted(k for k in label_map[p] if k not in observed[p])}
            for p in ("TCP", "UDP")},
    }
    (out / "label_mapping.json").write_text(json.dumps(mapping_doc, indent=2, ensure_ascii=False), encoding="utf-8")

    # ---- 7. Run manifest (reproducibility) ------------------------------------
    manifest = {"run_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
                "script": Path(__file__).name, "script_md5": md5_file(Path(__file__)),
                "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                "chunk_rows": CHUNK_ROWS, "zenodo": record_info,
                "schemas": {sid: {"n_columns": len(cols), "columns_raw": list(cols)} for cols, sid in schema_ids.items()}}
    (out / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print(f"\nDone. Outputs in {out}")


if __name__ == "__main__":
    main()


Writing h1_ait_verify.py


In [ ]:
# Integrity check of the written script (compare with the locally tested copy) + syntax check
import hashlib, py_compile
b = open('h1_ait_verify.py','rb').read()
print('bytes', len(b), 'md5', hashlib.md5(b).hexdigest(), '| md5 without trailing newlines', hashlib.md5(b.rstrip(b'\n')+b'\n').hexdigest())
py_compile.compile('h1_ait_verify.py', doraise=True); print('syntax OK')

bytes 25539 md5 d5eb4d04cd7beaf6728d5fe1573a72d5 | md5 without trailing newlines d5eb4d04cd7beaf6728d5fe1573a72d5
syntax OK


In [ ]:
# H-1 Step 6: full verification run (log kept as evidence in h1_outputs/run_log.txt)
!mkdir -p h1_outputs && python h1_ait_verify.py --data-dir ./sentinelai_ait_data --out-dir ./h1_outputs 2>&1 | tee h1_outputs/run_log.txt

[zenodo] AIT Netflow Data Set | doi 10.5281/zenodo.13168643 | 12 files listed
[archive] label_info.txt                 md5_match=True size_match=True zip=n/a -> verified=True
[archive] fox_netflows.zip               md5_match=True size_match=True zip=OK (all CRCs valid) -> verified=True
[archive] harrison_netflows.zip          md5_match=True size_match=True zip=OK (all CRCs valid) -> verified=True
[archive] russellmitchell_netflows.zip   md5_match=True size_match=True zip=OK (all CRCs valid) -> verified=True
[archive] santos_netflows.zip            md5_match=True size_match=True zip=OK (all CRCs valid) -> verified=True
[archive] shaw_netflows.zip              md5_match=True size_match=True zip=OK (all CRCs valid) -> verified=True
[archive] wardbeck_netflows.zip          md5_match=True size_match=True zip=OK (all CRCs valid) -> verified=True
[archive] wheeler_netflows.zip           md5_match=True size_match=True zip=OK (all CRCs valid) -> verified=True
[archive] wilson_netflows.zip     

In [ ]:
# H-1 Step 7: summarise the evidence files (read back from disk, nothing recomputed by hand)
import pandas as pd, json
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 120)
O = 'h1_outputs/'
nt = pd.read_csv(O+'ait_network_attack_counts.csv')
print(nt[['network','csv_files','total_flows','normal_flows','suspicious_flows','unmapped_flows','suspicious_pct','counts_reconcile']].to_string(index=False))
print()
print(nt[['network','tcp_complete_total_flows','tcp_complete_suspicious_flows','tcp_nocomplete_total_flows','tcp_nocomplete_suspicious_flows','udp_total_flows','udp_suspicious_flows']].to_string(index=False))
lab = pd.read_csv(O+'label_counts_by_file.csv', keep_default_na=False)
print('\nUNMAPPED LABEL ROWS:'); print(lab[lab.binary_label==''].to_string(index=False))
m = json.load(open(O+'label_mapping.json'))
print('\nlabel_info parse problems:', m['label_info_parse_problems'])
print('never observed TCP:', m['observed_in_csvs']['TCP']['official_labels_never_observed'])
print('never observed UDP:', m['observed_in_csvs']['UDP']['official_labels_never_observed'])

In [ ]:
# H-1 Step 8: quality summary per file + investigation of the 942 empty-label rows in shaw/tcp_complete.csv
import pandas as pd, json, zipfile, datetime as dt
fq = pd.read_csv('h1_outputs/csv_file_quality_report.csv', keep_default_na=False)
cols = ['network','file','rows','n_columns','schema_id','protocol_vs_schema_check','empty_cells_total','columns_with_empty_cells',
        'na_token_cells_total','exact_duplicate_rows','conflicting_label_groups','rows_in_conflicting_label_groups','parse_error']
print(fq[cols].to_string(index=False))
print('\nempty-cell columns (union):', sorted(set().union(*[json.loads(x).keys() for x in fq.empty_cells_by_column_json])))
print('na-token columns (union):', sorted(set().union(*[json.loads(x).keys() for x in fq.na_tokens_by_column_json])))
print('non-numeric cols by schema (first chunk):')
for sid, g in fq.groupby('schema_id'):
    print(' ', sid, g.mixed_or_text_columns_first_chunk_json.iloc[0])

In [ ]:
# H-1 Step 9: investigate the 942 empty-label rows in shaw/tcp_complete.csv (read-only, chunked, streamed from the ZIP)
import pandas as pd, zipfile, re
def clean(c): return re.sub(r':\d+$','',re.sub(r'^#(\d+#)?','',c))
keep = ['c_ip','c_port','s_ip','s_port','first','durat','c_pkts_all','s_pkts_all','http_res','fqdn','role_cli','role_serv','network_cli','network_serv','label']
empty, mal_t, http_vals = [], [], {}
with zipfile.ZipFile('sentinelai_ait_data/downloads/shaw_netflows.zip') as z, z.open('tcp_complete.csv') as f:
    for ch in pd.read_csv(f, dtype=str, keep_default_na=False, chunksize=200_000):
        ch.columns = [clean(c) for c in ch.columns]
        empty.append(ch.loc[ch.label=='', keep])
        for v,n in ch.http_res.value_counts().head(8).items(): http_vals[v] = http_vals.get(v,0)+n
        mal_t.append(pd.to_numeric(ch.loc[ch.label.isin(['service_scan','wpscan','dirb_scan','upload_rce_shell','online_cracking','vpn_connect']),'first']))
E = pd.concat(empty); M = pd.concat(mal_t)
print('empty-label rows:', len(E))
t = pd.to_datetime(pd.to_numeric(E['first']), unit='ms', utc=True)
print('empty-label time range :', t.min(), '->', t.max())
mt = pd.to_datetime(M, unit='ms', utc=True); print('selected attack-label time range:', mt.min(), '->', mt.max())
for col in ['s_port','c_ip','s_ip','role_cli','role_serv','network_cli','network_serv']:
    print(f'\n{col}:', E[col].value_counts().head(6).to_dict())
print('\nfirst 5 empty-label rows:'); print(E.head(5).to_string(index=False))
print('\nmost common http_res values (whole file, top):', dict(sorted(http_vals.items(), key=lambda kv:-kv[1])[:8]))

In [ ]:
# H-1 Step 10 (context only, no relabelling): how are OTHER shaw TCP flows to the same mail server on port 25 labelled?
import pandas as pd, zipfile, re
def clean(c): return re.sub(r':\d+$','',re.sub(r'^#(\d+#)?','',c))
parts = []
with zipfile.ZipFile('sentinelai_ait_data/downloads/shaw_netflows.zip') as z, z.open('tcp_complete.csv') as f:
    for ch in pd.read_csv(f, dtype=str, keep_default_na=False, chunksize=200_000):
        ch.columns = [clean(c) for c in ch.columns]
        sel = ch[(ch.s_ip=='10.70.34.215') & (ch.s_port=='25')]
        parts.append(sel.groupby(['c_ip','label'], dropna=False).size())
s = pd.concat(parts).groupby(level=[0,1]).sum().sort_values(ascending=False)
print(s.head(15).to_string())

In [ ]:
%%writefile h1_make_report.py
#!/usr/bin/env python3
"""
SentinelAI H-1 - build H1_AIT_Verification_Report.md from the evidence files
written by h1_ait_verify.py. Every number in the report is read from those files
(or recomputed here from the original ZIPs); nothing is typed in by hand.

Run:  python h1_make_report.py --data-dir ./sentinelai_ait_data --out-dir ./h1_outputs
"""
from __future__ import annotations

import argparse
import json
import re
import zipfile
from pathlib import Path

import pandas as pd

UNMAPPED_MAX_SHARE = 0.001      # GO criterion: unmapped flows must be < 0.1 % of all flows


def clean(c: str) -> str:
    """'#15#c_ip:1' -> 'c_ip' (same rule as the verifier)."""
    return re.sub(r":\d+$", "", re.sub(r"^#(\d+#)?", "", c))


def md_table(df: pd.DataFrame) -> str:
    """Minimal Markdown table writer (no extra dependency such as tabulate)."""
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        cells = []
        for v in r.tolist():
            if isinstance(v, bool):
                cells.append("yes" if v else "**NO**")
            elif isinstance(v, int) or (isinstance(v, float) and v.is_integer() and abs(v) >= 1000):
                cells.append(f"{int(v):,}")
            elif isinstance(v, float):
                cells.append(f"{v:.2f}")
            else:
                cells.append(str(v).replace("|", "\\|"))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


def pct(a, b):
    return 100.0 * a / b if b else float("nan")


def profile_unmapped(downloads: Path, lab: pd.DataFrame):
    """For every file that has unmapped labels, re-read it from the ZIP (chunked)
    and describe the unmapped rows plus how the SAME client/server/port tuples
    are labelled elsewhere in the file. Context only - nothing is relabelled."""
    out = []
    um = lab[lab["binary_label"] == ""]
    for (net, fname), g in um.groupby(["network", "file"]):
        bad_values = set(g["original_label"])
        rows, tuples = [], []
        with zipfile.ZipFile(downloads / f"{net}_netflows.zip") as z, z.open(fname) as f:
            for ch in pd.read_csv(f, dtype=str, keep_default_na=False, na_filter=False, chunksize=200_000):
                ch.columns = [clean(c) for c in ch.columns]
                rows.append(ch[ch["label"].isin(bad_values)])
                tuples.append(ch[["c_ip", "s_ip", "s_port", "label"]])
        R = pd.concat(rows)
        T = pd.concat(tuples)
        keys = R[["c_ip", "s_ip", "s_port"]].drop_duplicates()
        same = T.merge(keys, on=["c_ip", "s_ip", "s_port"])
        tcol = "first" if "first" in R.columns else "c_first_abs"
        t = pd.to_datetime(pd.to_numeric(R[tcol], errors="coerce"), unit="ms", utc=True)
        out.append({
            "network": net, "file": fname, "rows": len(R),
            "label_values": sorted(repr(v) for v in bad_values),
            "time_first": str(t.min()), "time_last": str(t.max()),
            "s_port": R["s_port"].value_counts().head(5).to_dict(),
            "c_ip": R["c_ip"].value_counts().head(5).to_dict(),
            "s_ip": R["s_ip"].value_counts().head(5).to_dict(),
            "role_serv": R["role_serv"].value_counts().head(5).to_dict() if "role_serv" in R else {},
            "same_tuple_labels": same["label"].map(lambda v: repr(v) if v in bad_values else v)
                                              .value_counts().to_dict(),
        })
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", default="./sentinelai_ait_data")
    ap.add_argument("--out-dir", default="./h1_outputs")
    args = ap.parse_args()
    downloads = Path(args.data_dir).resolve() / "downloads"
    O = Path(args.out_dir).resolve()

    arc = pd.read_csv(O / "archive_checksums.csv", keep_default_na=False)
    fq = pd.read_csv(O / "csv_file_quality_report.csv", keep_default_na=False)
    lab = pd.read_csv(O / "label_counts_by_file.csv", keep_default_na=False, dtype={"binary_label": str})
    nt = pd.read_csv(O / "ait_network_attack_counts.csv", keep_default_na=False)
    mapping = json.loads((O / "label_mapping.json").read_text())
    man = json.loads((O / "run_manifest.json").read_text())
    for c in ("rows", "normal_flows", "suspicious_flows", "unmapped_flows", "exact_duplicate_rows",
              "empty_cells_total", "na_token_cells_total", "conflicting_label_groups", "n_columns"):
        fq[c] = pd.to_numeric(fq[c])
    lab["flow_count"] = pd.to_numeric(lab["flow_count"])
    to_bool = lambda s: s.astype(str).str.lower().eq("true")
    for c in ("md5_match", "size_match", "verified"):
        arc[c] = to_bool(arc[c])
    fq["row_count_reconciles"] = to_bool(fq["row_count_reconciles"])
    fq["label_counts_sum_equals_rows"] = to_bool(fq["label_counts_sum_equals_rows"])
    nt["counts_reconcile"] = to_bool(nt["counts_reconcile"])

    zips = arc[arc["file"].str.endswith(".zip")]
    tot = nt[nt["network"] == "TOTAL"].iloc[0]
    nets = nt[nt["network"] != "TOTAL"].copy()
    total_flows = int(tot["total_flows"])
    unmapped_total = int(tot["unmapped_flows"])
    schemas = man["schemas"]

    # ------------------------------------------------------------ criteria
    crit = []
    crit.append(("All 8 archives and label_info.txt match the Zenodo MD5 and byte size, and every ZIP passes the CRC test",
                 bool(arc["verified"].all()) and len(zips) == 8))
    crit.append(("All 24 CSVs parsed with no errors and pandas row counts equal the physical line counts",
                 len(fq) == 24 and (fq["parse_error"] == "").all() and fq["row_count_reconciles"].all()))
    by_type = fq.groupby(["protocol", "flow_completeness"])["schema_id"].nunique()
    crit.append(("Each file type has one identical schema across all 8 networks, and every file has the label column",
                 (by_type == 1).all() and (fq["label_column"] == "label").all()
                 and (fq["protocol_vs_schema_check"] == "consistent").all()))
    crit.append(("label_info.txt parses into 4 sections with no conflicts, and every official label is observed",
                 mapping["label_info_parse_problems"] == []
                 and not mapping["observed_in_csvs"]["TCP"]["official_labels_never_observed"]
                 and not mapping["observed_in_csvs"]["UDP"]["official_labels_never_observed"]))
    crit.append((f"Unmapped flows are below {UNMAPPED_MAX_SHARE:.1%} of all flows and are reported, not guessed",
                 unmapped_total / total_flows < UNMAPPED_MAX_SHARE))
    crit.append(("Every network contains both Normal (0) and Suspicious (1) flows (needed for the H-6 network split)",
                 bool(((nets["normal_flows"].astype(int) > 0) & (nets["suspicious_flows"].astype(int) > 0)).all())))
    crit.append(("Per-network label counts sum to total flows (normal + suspicious + unmapped = rows)",
                 bool(nt["counts_reconcile"].all()) and fq["label_counts_sum_equals_rows"].all()))
    go = all(ok for _, ok in crit)

    unm = profile_unmapped(downloads, lab)

    # most common http_res values in the first chunk of one TCP-complete file (context for placeholders)
    http_top = {}
    with zipfile.ZipFile(downloads / "fox_netflows.zip") as zz, zz.open("tcp_complete.csv") as fh:
        first = next(pd.read_csv(fh, dtype=str, keep_default_na=False, na_filter=False, chunksize=200_000))
        first.columns = [clean(c) for c in first.columns]
        if "http_res" in first.columns:
            http_top = first["http_res"].value_counts().head(5).to_dict()

    # ------------------------------------------------------------ derived numbers
    by_kind = fq.groupby(["protocol", "flow_completeness"])[["rows", "normal_flows", "suspicious_flows", "unmapped_flows"]].sum()
    by_kind["suspicious_%"] = 100 * by_kind["suspicious_flows"] / by_kind["rows"]
    by_kind = by_kind.reset_index()
    nocomp = by_kind[(by_kind.protocol == "TCP") & (by_kind.flow_completeness == "nocomplete")].iloc[0]
    tcpc = by_kind[(by_kind.protocol == "TCP") & (by_kind.flow_completeness == "complete")].iloc[0]
    susp_share_from_nocomp = pct(nocomp["suspicious_flows"], int(tot["suspicious_flows"]))
    dup_total = int(fq["exact_duplicate_rows"].sum())
    dup_by_proto = fq.groupby("protocol")["exact_duplicate_rows"].sum().to_dict()
    conflicts = int(fq["conflicting_label_groups"].sum())
    na_cols = sorted(set().union(*[json.loads(x).keys() for x in fq["na_tokens_by_column_json"]]))
    na_total = int(fq["na_token_cells_total"].sum())
    empty_cols = sorted(set().union(*[json.loads(x).keys() for x in fq["empty_cells_by_column_json"]]))
    empty_total = int(fq["empty_cells_total"].sum())
    nets["suspicious_pct"] = pd.to_numeric(nets["suspicious_pct"])
    lo = nets.loc[nets["suspicious_pct"].idxmin()]
    hi = nets.loc[nets["suspicious_pct"].idxmax()]

    mal_by_net = lab[lab["binary_label"] == "1"].pivot_table(index="original_label", columns="network",
                                                              values="flow_count", aggfunc="sum", fill_value=0)

    # time span per network (TCP 'first' and UDP 'c_first_abs' are epoch milliseconds)
    fq["time_min"] = pd.to_numeric(fq["time_min"], errors="coerce")
    fq["time_max"] = pd.to_numeric(fq["time_max"], errors="coerce")
    span = fq.groupby("network").agg(t0=("time_min", "min"), t1=("time_max", "max"))
    span["first_flow_utc"] = pd.to_datetime(span["t0"], unit="ms", utc=True).dt.strftime("%Y-%m-%d %H:%M")
    span["last_flow_utc"] = pd.to_datetime(span["t1"], unit="ms", utc=True).dt.strftime("%Y-%m-%d %H:%M")
    span["days"] = ((span["t1"] - span["t0"]) / 86_400_000).round(2)

    # ------------------------------------------------------------ write report
    L = []
    w = L.append
    w("# H-1 — AIT Netflow Data Set Verification Report")
    w("")
    w(f"*SentinelAI · Data & Features · generated by `h1_make_report.py` from the evidence files of the "
      f"verification run at {man['run_utc']} (UTC).*")
    w("")
    w(f"**Recommendation: {'GO' if go else 'NO-GO'}"
      f"{' — with the conditions in section 10' if go else ' — see the failed criteria in section 10'}.**")
    w("")
    w("## 1. Objective")
    w("")
    w("Task H-1 (tracker): download AIT Netflow (8 networks); confirm the CSV columns and `label_info.txt`; "
      "count attack flows per network. Done when there is a column table, an attack-count table and a GO/NO-GO "
      "decision on AIT (the Phase 1 gate). The project uses binary labels: **Normal = 0, Suspicious = 1**.")
    w("")
    w("## 2. Dataset source")
    w("")
    z = man["zenodo"]
    w(f"- **{z['title']}**, AIT Austrian Institute of Technology, Zenodo record {z['record_id']} "
      f"(DOI [{z['doi']}](https://doi.org/{z['doi']})), published {z['publication_date']}.")
    w(f"- Official file list, sizes and MD5 checksums read from the Zenodo REST API (`{z['api_url']}`) "
      f"at {z['retrieved_utc']}.")
    w("- Files used: `label_info.txt` and `<network>_netflows.zip` for fox, harrison, russellmitchell, santos, "
      "shaw, wardbeck, wheeler and wilson. Each ZIP holds `tcp_complete.csv`, `tcp_nocomplete.csv` and "
      "`udp_complete.csv` (Tstat flow logs with labels added by the publisher).")
    w("")
    w("## 3. Verification methodology")
    w("")
    w(f"Run environment: Google Colab, Python {man['python']}, pandas {man['pandas']}, numpy {man['numpy']}. "
      f"Verifier `h1_ait_verify.py` (MD5 `{man['script_md5']}`), chunk size {man['chunk_rows']:,} rows.")
    w("")
    w("1. **Archives**: the verifier computed the MD5 and byte size of every downloaded file and compared them with the values "
      "Zenodo lists. It ran `zipfile.testzip()`, which reads every member and checks its CRC-32. "
      "All files are checked, and a failure is recorded rather than stopping the run.")
    w("2. **CSVs**: each CSV was streamed straight out of its ZIP with `pandas.read_csv(chunksize=200000, dtype=str, "
      "na_filter=False, on_bad_lines='error')`. The archives were never modified and only one chunk was in memory at a time. "
      "Newlines were counted in the same pass, so the pandas row count can be reconciled against the physical line count.")
    w("3. **Labels**: `label_info.txt` was parsed into its four sections (Benign/Malicious × TCP/UDP). The mapping is "
      "**exact** (case- and whitespace-sensitive) and **protocol-aware**: TCP files use the TCP sections and UDP files "
      "use the UDP sections. Any other value is *unmapped* and is never counted as benign.")
    w("4. **Quality**: per file, the verifier counted empty cells and NA-like tokens (`NA`, `N/A`, `nan`, `NaN`, `null`, `NULL`, "
      "`None`, `-`), exact duplicate rows, and groups of rows with identical features but different binary labels "
      "(both via 64-bit row hashes over the full file). It also cross-checked the protocol in each filename against "
      "the schema and recorded the timestamp range.")
    w("5. **Report**: this file is generated from the CSV/JSON outputs by `h1_make_report.py`. The unmapped-row profile in "
      "section 7 is recomputed from the original ZIP.")
    w("")
    w("### Corrections to the original `sentinelai_ait_verify.py`")
    w("")
    rm = arc[arc["file"] == "russellmitchell_netflows.zip"].iloc[0]
    diffpos = [i + 1 for i, (x, y) in enumerate(zip(rm["script_v1_expected_md5"], rm["official_md5"])) if x != y]
    others_ok = arc[(arc["file"] != "russellmitchell_netflows.zip") & arc["file"].str.endswith(".zip")]
    others_ok = (others_ok["script_v1_md5_equals_official"].astype(str).str.lower() == "true").all()
    w(f"- **Checksum transcription error (line 32).** The script expected `{rm['script_v1_expected_md5']}`, "
      f"but Zenodo lists `{rm['official_md5']}` and the downloaded file hashes to `{rm['actual_md5']}`. "
      + (f"They differ only at character positions {diffpos}: two characters were swapped "
         f"(`{rm['script_v1_expected_md5'][diffpos[0]-1:diffpos[1]]}` instead of `{rm['official_md5'][diffpos[0]-1:diffpos[1]]}`). "
         if len(diffpos) == 2 and sorted(rm['script_v1_expected_md5']) == sorted(rm['official_md5'])
         else f"They differ at character positions {diffpos}. ")
      + f"{'The archive was correct and the script was wrong. ' if rm['md5_match'] else ''}"
      "The constant was corrected with a comment, and the untouched original is kept as `sentinelai_ait_verify.ORIGINAL.py`. "
      f"{'The other seven hard-coded MD5s equal the official values.' if others_ok else 'Other hard-coded MD5s also differ; see section 4.'}")
    w("- **Abort on first mismatch.** The original raised an exception at the first failing archive, so no output files "
      "were written and the later archives were never reported. v2 checks and records every archive.")
    w("- **Whole-file reads.** The original loaded each CSV fully with `pd.read_csv`. v2 streams in chunks.")
    w("- **Non-exact label matching.** The original applied `.str.strip()` before mapping, and pandas' default NA parsing "
      "can silently turn text such as `NA` or `null` into missing values. v2 matches exactly, reports "
      "whitespace-only variants separately and disables NA conversion.")
    w("- **Checksums not read from the publisher; sizes not checked.** v2 reads both from the Zenodo API at run time.")
    w("")
    w("## 4. Archive verification results")
    w("")
    a = arc[["file", "official_bytes", "actual_bytes", "official_md5", "actual_md5", "md5_match", "size_match",
             "zip_testzip", "verified"]].copy()
    a["official_bytes"] = pd.to_numeric(a["official_bytes"]); a["actual_bytes"] = pd.to_numeric(a["actual_bytes"])
    a["zip_testzip"] = a["zip_testzip"].replace("", "n/a (text file)")
    w(md_table(a))
    w("")
    w(f"Result: **{int(arc['verified'].sum())} of {len(arc)} files verified**; "
      f"{'no failed or unresolved archive checks' if arc['verified'].all() else 'see rows marked NO'}.")
    w("")
    w("## 5. CSV structure and columns")
    w("")
    sch_rows = []
    for sid, g in fq.groupby("schema_id"):
        sch_rows.append({"schema": sid, "file": ", ".join(sorted(g["file"].unique())),
                         "protocol": ", ".join(sorted(g["protocol"].unique())),
                         "networks": g["network"].nunique(), "columns": int(g["n_columns"].iloc[0]),
                         "label column": ", ".join(sorted(g["label_column"].unique())),
                         "protocol vs schema": ", ".join(sorted(g["protocol_vs_schema_check"].unique()))})
    w(md_table(pd.DataFrame(sch_rows)))
    w("")
    w(f"There are {fq['schema_id'].nunique()} distinct header layouts. Files of the same type share one layout across "
      f"{'all 8 networks' if (by_type == 1).all() else 'SOME networks only (inconsistent)'}; the layouts of different file types "
      "differ, so the schemas are **not identical across file types**. Header names are Tstat-style "
      "(`#15#c_ip:1`, `c_port:2`, …): a `#…#` prefix on the first field and a `:N` position suffix on most "
      "fields. Below, `clean name` removes those decorations; the exact raw names are in "
      "`csv_file_quality_report.csv` (`columns_raw_json`) and `run_manifest.json`.")
    w("")
    for cols_raw, sid in sorted(((tuple(v["columns_raw"]), k) for k, v in schemas.items()), key=lambda x: x[1]):
        files = ", ".join(sorted(fq[fq["schema_id"] == sid]["file"].unique()))
        w(f"<details><summary><b>{sid}</b> — {files}: {len(cols_raw)} columns (click to expand)</summary>")
        w("")
        w("| # | raw header | clean name |")
        w("|---|---|---|")
        for i, c in enumerate(cols_raw, 1):
            w(f"| {i} | `{c}` | `{clean(c)}` |")
        w("")
        w("</details>")
        w("")
    w("### Per-file inspection")
    w("")
    f = fq[["network", "file", "rows", "n_columns", "schema_id", "label_column", "row_count_reconciles",
            "empty_cells_total", "na_token_cells_total", "exact_duplicate_rows", "conflicting_label_groups",
            "parse_error"]].copy()
    f["parse_error"] = f["parse_error"].replace("", "none")
    w(md_table(f))
    w("")
    w("## 6. Official label mapping (`label_info.txt`)")
    w("")
    w(f"Source MD5 `{mapping['source_md5']}` (matches Zenodo). Parse problems: "
      f"{mapping['label_info_parse_problems'] or 'none'}. Mapping file: `label_mapping.json`.")
    w("")
    for p in ("TCP", "UDP"):
        w(f"- **{p} benign → 0** ({len(mapping[p]['benign_0'])}): " + ", ".join(f"`{x}`" for x in mapping[p]["benign_0"]))
        w(f"- **{p} malicious → 1** ({len(mapping[p]['malicious_1'])}): " + ", ".join(f"`{x}`" for x in mapping[p]["malicious_1"]))
    w("")
    never = {p: mapping["observed_in_csvs"][p]["official_labels_never_observed"] for p in ("TCP", "UDP")}
    w("Observed in the CSVs: " + ("every official label occurs at least once. " if not (never["TCP"] or never["UDP"])
      else f"official labels never observed: {never}. ")
      + "Labels that occur but are not in `label_info.txt` for that protocol:")
    w("")
    um_lab = lab[lab["binary_label"] == ""][["network", "file", "protocol", "original_label", "official_category", "flow_count"]].copy()
    um_lab["original_label"] = um_lab["original_label"].map(repr)
    w(md_table(um_lab) if len(um_lab) else "_none_")
    w("")
    w("## 7. Attack counts per network")
    w("")
    w("Counts include **all three files per network**: complete TCP flows, non-complete TCP flows and UDP flows. "
      "*Suspicious* means the label is in a *Malicious* section of `label_info.txt`; *unmapped* rows are not in either class.")
    w("")
    t1 = nt[["network", "total_flows", "normal_flows", "suspicious_flows", "unmapped_flows", "suspicious_pct",
             "tcp_total_flows", "tcp_suspicious_flows", "udp_total_flows", "udp_suspicious_flows", "counts_reconcile"]].copy()
    for c in t1.columns[1:-1]:
        t1[c] = pd.to_numeric(t1[c])
    t1 = t1.rename(columns={"suspicious_pct": "suspicious %"})
    w(md_table(t1))
    w("")
    w("Split by file type:")
    w("")
    t2 = nt[["network", "tcp_complete_total_flows", "tcp_complete_suspicious_flows", "tcp_nocomplete_total_flows",
             "tcp_nocomplete_suspicious_flows", "udp_total_flows", "udp_suspicious_flows"]].copy()
    for c in t2.columns[1:]:
        t2[c] = pd.to_numeric(t2[c])
    w(md_table(t2))
    w("")
    w("Totals by file type:")
    w("")
    w(md_table(by_kind.rename(columns={"flow_completeness": "completeness"})))
    w("")
    w("Suspicious flows by original attack label (all networks):")
    w("")
    mt = mal_by_net.copy()
    mt["TOTAL"] = mt.sum(axis=1)
    mt = mt.sort_values("TOTAL", ascending=False).reset_index().rename(columns={"original_label": "label"})
    w(md_table(mt))
    w("")
    if unm:
        w("### Unmapped rows — investigation (context only, not relabelled)")
        w("")
        for u in unm:
            w(f"- **{u['network']}/{u['file']}**: {u['rows']:,} rows with label value(s) {', '.join(u['label_values'])}. "
              f"Server port(s) {u['s_port']}; client IPs {u['c_ip']}; server IP {u['s_ip']}; server role {u['role_serv']}. "
              f"Time range {u['time_first']} → {u['time_last']}.")
            w(f"  - Labels of all rows in the same file with the same (client IP, server IP, server port): {u['same_tuple_labels']}.")
            mal_labels = set(mapping["TCP"]["malicious_1"]) | set(mapping["UDP"]["malicious_1"])
            if any(k in mal_labels for k in u["same_tuple_labels"]):
                w("  - Interpretation: the same client/server/port combinations also carry a **malicious** label, so treating "
                  "these rows as benign would be an unsupported assumption. They stay unmapped.")
            else:
                w("  - Interpretation: there is no evidence that fixes their class, so they stay unmapped.")
        w("")
    w("## 8. Data quality findings")
    w("")
    w("**Verified findings**")
    w("")
    w(f"- Missing values: {empty_total:,} empty cells in the whole dataset, all in column(s) {empty_cols}. These are the "
      f"{'unmapped rows in section 7. No feature column has empty cells.' if empty_cols in ([], ['label']) else 'see csv_file_quality_report.csv for the per-column breakdown.'}")
    w(f"- Placeholder tokens: {na_total:,} cells hold an NA-like token, all in the text columns {na_cols} "
      "(Tstat writes `-` when a field does not apply, e.g. no TLS SNI). "
      + (f"`http_res` mixes numeric HTTP status codes with text; its most common values in the first "
         f"200,000 rows of fox/tcp_complete.csv are {http_top}. " if http_top else "")
      + "These are placeholders, not corruption. Feature engineering (H-4) must handle them explicitly.")
    w(f"- Duplicates: {dup_total:,} exact duplicate rows in total (by protocol: {dup_by_proto}), i.e. "
      f"{pct(dup_total, total_flows):.4f}% of flows. "
      f"There are {conflicts} groups of rows with identical features but different binary labels.")
    w(f"- Columns: {'each file type has one consistent schema across networks' if (by_type == 1).all() else 'schemas are NOT consistent across networks'}. "
      f"There are {fq['schema_id'].nunique()} distinct schemas in total (see section 5).")
    w(f"- Unexpected labels: {unmapped_total:,} unmapped flows ({pct(unmapped_total, total_flows):.4f}% of all flows), "
      "only in the file(s) listed in section 6. "
      + ("No label appeared in the wrong protocol's list." if not lab["official_category"].str.contains("listed under").any()
         else "Some labels appear in the wrong protocol's list (see section 6)."))
    w(f"- Class balance: overall {pct(int(tot['suspicious_flows']), total_flows):.2f}% suspicious. "
      f"By file type: TCP-complete {tcpc['suspicious_%']:.2f}%, TCP-non-complete {nocomp['suspicious_%']:.2f}%, "
      f"UDP {by_kind[by_kind.protocol == 'UDP']['suspicious_%'].iloc[0]:.2f}%. "
      f"Non-complete TCP flows account for {susp_share_from_nocomp:.1f}% of all suspicious flows.")
    w(f"- Differences between networks: the suspicious share ranges from {lo['suspicious_pct']:.2f}% ({lo['network']}) "
      f"to {hi['suspicious_pct']:.2f}% ({hi['network']}). "
      + ("Every network contains both classes." if crit[5][1] else "NOT every network contains both classes."))
    w("")
    w("Capture period per network (from the `first` / `c_first_abs` epoch-millisecond columns):")
    w("")
    w(md_table(span[["first_flow_utc", "last_flow_utc", "days"]].reset_index()))
    w("")
    w("**Assumptions**")
    w("")
    w("- The publisher's labels are taken as ground truth. H-1 checked that they are consistent and complete, not that each "
      "label is correct.")
    w("- The protocol comes from the file name (`tcp_*` / `udp_*`) and was cross-checked against the schema "
      "(TCP files contain `c_syn_cnt`, `c_fin_cnt` and `c_rst_cnt`; UDP files do not).")
    w("- Duplicate and conflict detection relies on 64-bit hashes of full rows. The collision probability at this "
      "scale is negligible but not zero.")
    w("")
    w("**Recommendations for later tasks** (not done in H-1)")
    w("")
    w(f"- H-3: exclude the {unmapped_total:,} unmapped flows (or ask the publisher) and state this in the labelling rule. "
      "Never map them to 0.")
    if nocomp["suspicious_%"] > 90:
        w(f"- H-3/H-4: decide explicitly whether `tcp_nocomplete` flows are used. They are {nocomp['suspicious_%']:.1f}% "
          "suspicious, so a model that can tell which file a flow came from (or can see the features that define "
          "‘non-complete’) gets a near-trivial shortcut. Report results with and without them.")
    w(f"- H-4: the {fq['schema_id'].nunique()} schemas need either a common feature subset or per-protocol handling. Treat `-`/`---` "
      "placeholders as categorical ‘not applicable’ values. Drop IPs and timestamps, as the tracker already requires.")
    w(f"- M-3/H-6: the classes are imbalanced and vary by network ({lo['suspicious_pct']:.1f}%–{hi['suspicious_pct']:.1f}%). "
      "Use class weights and report per-network metrics.")
    w("")
    w("## 9. Limitations")
    w("")
    w("- MD5 proves the files are identical to what Zenodo publishes. It does not protect against a malicious publisher, "
      "and MD5 is not collision-resistant. SHA-256 was not available from the record.")
    w("- The numeric type profile (`mixed_or_text_columns_first_chunk_json`) covers only the first 200,000 rows of each "
      "file. Empty-cell, NA-token, duplicate and label counts cover every row.")
    w("- Only exact duplicates *within* a file were counted; cross-file and cross-network duplicates were not.")
    w("- H-1 did not check whether labels are correct in time against the attack schedule; that belongs to label "
      "auditing (H-3) and Reem's R-6.")
    w("")
    w("## 10. GO / NO-GO")
    w("")
    w("Criteria applied (set by this verification, not defined in the tracker):")
    w("")
    for txt, ok in crit:
        w(f"- [{'x' if ok else ' '}] {txt} — **{'PASS' if ok else 'FAIL'}**")
    w("")
    if go:
        w(f"**Decision: GO.** The AIT Netflow data is verified, complete and labelled well enough for the "
          f"Normal(0)/Suspicious(1) task. {total_flows - unmapped_total:,} of {total_flows:,} flows map to a binary "
          "label, and every network has attacks for a network-wise split. Conditions carried into Phase 2:")
        w("")
        w(f"1. Exclude (do not relabel) the {unmapped_total:,} unmapped flows, and document this in H-3.")
        k = 2
        if nocomp["suspicious_%"] > 90:
            k += 1
            w("2. Make an explicit, documented decision on `tcp_nocomplete` flows because of the label shortcut described in section 8.")
        w(f"{k}. Handle the {fq['schema_id'].nunique()} different schemas and the placeholder tokens in the feature contract (ALL-1/H-4).")
    else:
        w("**Decision: NO-GO.** One or more criteria failed (unchecked above). Resolve them before Phase 2, or switch to the "
          "CSE-CIC-IDS2018 backup (H-2).")
    w("")
    w("## Evidence files")
    w("")
    for fn in ("archive_checksums.csv", "csv_file_quality_report.csv", "label_counts_by_file.csv",
               "ait_network_attack_counts.csv", "label_mapping.json", "run_manifest.json", "run_log.txt"):
        w(f"- `{fn}`")
    w("- Scripts: `h1_ait_verify.py`, `h1_make_report.py`. Original script kept as `sentinelai_ait_verify.ORIGINAL.py`.")
    w("")
    (O / "H1_AIT_Verification_Report.md").write_text("\n".join(L), encoding="utf-8")
    print("GO" if go else "NO-GO")
    for txt, ok in crit:
        print(("PASS " if ok else "FAIL ") + txt)
    print("written", O / "H1_AIT_Verification_Report.md")


if __name__ == "__main__":
    main()


Writing h1_make_report.py


In [ ]:
# Integrity check of the report generator (must equal the locally tested copy: b448dfa2f682aaa98be7dce7c6fc6203)
import hashlib, py_compile
print('md5', hashlib.md5(open('h1_make_report.py','rb').read()).hexdigest()); py_compile.compile('h1_make_report.py', doraise=True); print('syntax OK')

md5 b448dfa2f682aaa98be7dce7c6fc6203
syntax OK


In [ ]:
# H-1 Step 11: generate the Markdown report from the evidence files, then list all deliverables with their MD5
!python h1_make_report.py --data-dir ./sentinelai_ait_data --out-dir ./h1_outputs 2>&1 | tee -a h1_outputs/run_log.txt
!cp h1_ait_verify.py h1_make_report.py sentinelai_ait_verify.py sentinelai_ait_verify.ORIGINAL.py h1_outputs/ && cd h1_outputs && md5sum * && ls -la

GO
PASS All 8 archives and label_info.txt match the Zenodo MD5 and byte size, and every ZIP passes the CRC test
PASS All 24 CSVs parsed with no errors and pandas row counts equal the physical line counts
PASS Each file type has one identical schema across all 8 networks, and every file has the label column
PASS label_info.txt parses into 4 sections with no conflicts, and every official label is observed
PASS Unmapped flows are below 0.1% of all flows and are reported, not guessed
PASS Every network contains both Normal (0) and Suspicious (1) flows (needed for the H-6 network split)
PASS Per-network label counts sum to total flows (normal + suspicious + unmapped = rows)
written /content/h1_outputs/H1_AIT_Verification_Report.md
cp: cannot stat 'sentinelai_ait_verify.py': No such file or directory
cp: cannot stat 'sentinelai_ait_verify.ORIGINAL.py': No such file or directory


In [ ]:
# Review: print the report without the long collapsed column lists (those are in the file itself)
import re
rep = open('h1_outputs/H1_AIT_Verification_Report.md').read()
print(len(rep), 'chars;', rep.count('<details>'), 'collapsed column tables')
print(re.sub(r'<details>.*?</details>', '[column table]', rep, flags=re.S))

In [ ]:
# H-1 Step 12: independent reconciliation across the four evidence files (all must print True)
import pandas as pd, json
O='h1_outputs/'
fq=pd.read_csv(O+'csv_file_quality_report.csv'); lab=pd.read_csv(O+'label_counts_by_file.csv', keep_default_na=False)
nt=pd.read_csv(O+'ait_network_attack_counts.csv'); arc=pd.read_csv(O+'archive_checksums.csv')
tot=nt[nt.network=='TOTAL'].iloc[0]
zipbytes=sum(m['uncompressed_bytes'] for z in arc.zip_members.dropna() if z for m in json.loads(z))
print('1 label counts sum == rows (all files):', lab.flow_count.sum()==fq.rows.sum()==tot.total_flows)
print('2 rows == physical lines - 1 (all files):', (fq.rows==fq.physical_lines_incl_header-1).all())
print('3 per network label sums == quality rows == attack-count totals:',
      (lab.groupby('network').flow_count.sum()==fq.groupby('network').rows.sum()).all() and
      (fq.groupby('network').rows.sum()==nt[nt.network!='TOTAL'].set_index('network').total_flows).all())
print('4 normal+suspicious+unmapped == total:', tot.normal_flows+tot.suspicious_flows+tot.unmapped_flows==tot.total_flows)
print('5 bytes streamed == ZIP uncompressed sizes:', fq.uncompressed_bytes_read.sum()==zipbytes)
print('6 binary sums from label file:', {k:int(v) for k,v in lab.groupby('binary_label').flow_count.sum().items()})
print('TOTAL:', {k:(int(tot[k]) if k!='suspicious_pct' else tot[k]) for k in ['total_flows','normal_flows','suspicious_flows','unmapped_flows','suspicious_pct','tcp_total_flows','udp_total_flows']})

In [ ]:
# Bundle all H-1 deliverables for download (Files panel -> h1_outputs.zip). Original archives are untouched.
!rm -f h1_outputs.zip && zip -qr h1_outputs.zip h1_outputs && ls -la h1_outputs.zip && unzip -l h1_outputs.zip | tail -n 16

In [ ]:
# ===================== H-2: CSE-CIC-IDS2018 backup dataset verification =====================
# Step 1: list the OFFICIAL public bucket s3://cse-cic-ids2018 (linked from unb.ca/cic/datasets/ids-2018.html)
# using the anonymous S3 REST API (no AWS account, no download of data). Saves the full listing as evidence.
import requests, xml.etree.ElementTree as ET, pandas as pd, os, datetime as dt
os.makedirs('h2_outputs', exist_ok=True)
BUCKET_URL = 'https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/'
NS = {'s3': 'http://s3.amazonaws.com/doc/2006-03-01/'}
rows, token = [], None
while True:
    params = {'list-type': '2', 'max-keys': '1000'}
    if token: params['continuation-token'] = token
    r = requests.get(BUCKET_URL, params=params, timeout=60); r.raise_for_status()
    root = ET.fromstring(r.content)
    for o in root.findall('s3:Contents', NS):
        rows.append({'key': o.find('s3:Key', NS).text, 'bytes': int(o.find('s3:Size', NS).text),
                     'last_modified': o.find('s3:LastModified', NS).text, 'etag': o.find('s3:ETag', NS).text.strip('"')})
    if root.find('s3:IsTruncated', NS).text != 'true': break
    token = root.find('s3:NextContinuationToken', NS).text
lst = pd.DataFrame(rows)
lst.to_csv('h2_outputs/s3_bucket_listing.csv', index=False)
print('listed at', dt.datetime.utcnow().isoformat(timespec='seconds'), 'UTC | objects:', len(lst), '| total GiB:', round(lst.bytes.sum()/2**30, 2))
lst['top'] = lst.key.str.split('/').str[0]
print(lst.groupby('top').agg(objects=('key','size'), GiB=('bytes', lambda b: round(b.sum()/2**30, 2))).to_string())
print()
ml = lst[lst.key.str.contains('Processed Traffic Data for ML')]
print(ml[['key','bytes','last_modified']].to_string(index=False))

listed at 2026-11-25T18:55:40 UTC | objects: 42 | total GiB: 452.75
                                          objects     GiB
top                                                      
Original Network Traffic and Log data          31  446.34
Processed Traffic Data for ML Algorithms       11    6.41

                                                                                        key      bytes            last_modified
                                                  Processed Traffic Data for ML Algorithms/          0 2018-10-11T16:02:25.000Z
   Processed Traffic Data for ML Algorithms/Friday-02-03-2018_TrafficForML_CICFlowMeter.csv  352368373 2018-10-11T16:02:49.000Z
   Processed Traffic Data for ML Algorithms/Friday-16-02-2018_TrafficForML_CICFlowMeter.csv  333723605 2018-10-11T16:03:10.000Z
   Processed Traffic Data for ML Algorithms/Friday-23-02-2018_TrafficForML_CICFlowMeter.csv  382840456 2018-10-11T16:03:33.000Z
 Processed Traffic Data for ML Algorithms/Thuesday-20-02-20

/tmp/ipykernel_1838/2838151432.py:21: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print('listed at', dt.datetime.utcnow().isoformat(timespec='seconds'), 'UTC | objects:', len(lst), '| total GiB:', round(lst.bytes.sum()/2**30, 2))


In [ ]:
# Step 1b: what is in the raw part of the bucket (names and sizes only)
raw = lst[lst.top=='Original Network Traffic and Log data'].copy()
raw['GiB'] = (raw.bytes/2**30).round(2)
print(raw[['key','GiB']].to_string(index=False).replace('Original Network Traffic and Log data/','.../'))

                                                                key   GiB
                             .../  0.00
           .../Friday-02-03-2018/  0.00
   .../Friday-02-03-2018/logs.zip  0.22
   .../Friday-02-03-2018/pcap.zip 41.71
           .../Friday-16-02-2018/  0.00
   .../Friday-16-02-2018/logs.zip  0.14
   .../Friday-16-02-2018/pcap.zip 35.89
           .../Friday-23-02-2018/  0.00
   .../Friday-23-02-2018/logs.zip  0.20
   .../Friday-23-02-2018/pcap.zip 55.04
         .../Thursday-01-03-2018/  0.00
 .../Thursday-01-03-2018/logs.zip  0.21
 .../Thursday-01-03-2018/pcap.zip 48.76
         .../Thursday-15-02-2018/  0.00
 .../Thursday-15-02-2018/logs.zip  0.14
 .../Thursday-15-02-2018/pcap.zip 38.45
         .../Thursday-22-02-2018/  0.00
 .../Thursday-22-02-2018/logs.zip  0.19
 .../Thursday-22-02-2018/pcap.zip 46.79
          .../Tuesday-20-02-2018/  0.00
  .../Tuesday-20-02-2018/logs.zip  0.17
  .../Tuesday-20-02-2018/pcap.rar 41.34
        .../Wednesday-14-02-2018/  0.00
.../We

In [ ]:
# Step 2: read ONLY the first 64 KB of each processed CSV (HTTP Range request) to compare headers across days.
import requests, json, urllib.parse
csv_keys = sorted(k for k in lst.key if k.endswith('.csv'))
def url(k): return BUCKET_URL + urllib.parse.quote(k)
headers = {}
for k in csv_keys:
    b = requests.get(url(k), headers={'Range': 'bytes=0-65535'}, timeout=60).content
    lines = b.decode('utf-8', 'replace').split('\n')
    hdr = lines[0].rstrip('\r').split(',')
    headers[k.split('/')[-1]] = hdr
    print(f"{k.split('/')[-1]:<52} fields={len(hdr):>3}  CRLF={'\\r' in lines[0]}  first-row fields={len(lines[1].split(','))}")
base = headers['Friday-02-03-2018_TrafficForML_CICFlowMeter.csv']
print('\nReference header (Friday-02-03):', len(base), 'columns'); print(base)
for f, h in headers.items():
    if h != base:
        print('\nDIFFERS:', f, '| extra:', [c for c in h if c not in base], '| missing:', [c for c in base if c not in h],
              '| same order for shared columns:', [c for c in h if c in base] == base)
json.dump(headers, open('h2_outputs/csv_headers.json','w'), indent=1)

Friday-02-03-2018_TrafficForML_CICFlowMeter.csv      fields= 80  CRLF=False  first-row fields=80
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv      fields= 80  CRLF=False  first-row fields=80
Friday-23-02-2018_TrafficForML_CICFlowMeter.csv      fields= 80  CRLF=False  first-row fields=80
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv    fields= 84  CRLF=False  first-row fields=84
Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv    fields= 80  CRLF=False  first-row fields=80
Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv    fields= 80  CRLF=False  first-row fields=80
Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv    fields= 80  CRLF=False  first-row fields=80
Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv   fields= 80  CRLF=False  first-row fields=80
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv   fields= 80  CRLF=False  first-row fields=80
Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv   fields= 80  CRLF=False  first-row fields=80

Reference header (Friday-02-0

In [ ]:
%%writefile h2_ids2018_scan.py
#!/usr/bin/env python3
"""
SentinelAI H-2 - CSE-CIC-IDS2018 (backup dataset) verification scan.

Streams each of the 10 official "Processed Traffic Data for ML Algorithms" CSVs
from the public bucket s3://cse-cic-ids2018 (anonymous HTTPS) in chunks. Nothing
is written to disk except the summary outputs. For every file it records:
  * exact header, number of rows, physical line count (reconciliation)
  * repeated header lines inside the data (rows whose Label == "Label")
  * exact Label value counts (no stripping / case folding)
  * full-file checks on selected numeric columns (empty, non-numeric, inf, negative)
  * Timestamp range and Protocol values
and, on the first SAMPLE_ROWS rows of each file only, a full-column profile
(empty / non-numeric / infinite / negative cells, exact duplicate rows, constant columns).

Run: python h2_ids2018_scan.py --out-dir ./h2_outputs
"""
from __future__ import annotations

import argparse
import io
import json
import time
import urllib.parse
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests

BUCKET_URL = "https://cse-cic-ids2018.s3.ca-central-1.amazonaws.com/"
PREFIX = "Processed Traffic Data for ML Algorithms/"
CHUNK = 250_000
SAMPLE_ROWS = 200_000
# Columns checked over EVERY row (cheap). All exist in both header variants.
FULL_COLS = ["Dst Port", "Protocol", "Timestamp", "Flow Duration", "Flow Byts/s", "Flow Pkts/s",
             "Init Fwd Win Byts", "Label"]
NUMERIC_FULL = ["Dst Port", "Protocol", "Flow Duration", "Flow Byts/s", "Flow Pkts/s", "Init Fwd Win Byts"]


class LineCountingReader(io.RawIOBase):
    """Counts newline bytes as pandas reads the stream (row-count reconciliation)."""
    def __init__(self, raw):
        self.raw, self.newlines, self.nbytes, self.last = raw, 0, 0, b""
    def readable(self):
        return True
    def readinto(self, b):
        data = self.raw.read(len(b))
        n = len(data)
        if n:
            b[:n] = data
            self.newlines += data.count(b"\n"); self.nbytes += n; self.last = data[-1:]
        return n
    def physical_lines(self):
        return 0 if not self.nbytes else self.newlines + (0 if self.last == b"\n" else 1)


def list_csvs():
    """Anonymous S3 ListObjectsV2 for the processed-CSV prefix."""
    import xml.etree.ElementTree as ET
    ns = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
    r = requests.get(BUCKET_URL, params={"list-type": "2", "prefix": PREFIX}, timeout=60)
    r.raise_for_status()
    root = ET.fromstring(r.content)
    out = []
    for o in root.findall("s3:Contents", ns):
        k = o.find("s3:Key", ns).text
        if k.endswith(".csv"):
            out.append({"key": k, "bytes": int(o.find("s3:Size", ns).text),
                        "etag": o.find("s3:ETag", ns).text.strip('"')})
    return sorted(out, key=lambda d: d["key"])


def numeric_issues(s: pd.Series):
    """Counts for one string column: empty, non-numeric (excl. inf), +/-inf, negative."""
    empty = int((s == "").sum())
    low = s.str.lower()
    inf = int(low.isin(["inf", "infinity", "-inf", "-infinity"]).sum())
    num = pd.to_numeric(s.where(~low.isin(["inf", "infinity", "-inf", "-infinity"])), errors="coerce")
    nonnum = int(num.isna().sum()) - empty - inf
    neg = int((num < 0).sum())
    return empty, nonnum, inf, neg


def scan_file(key: str, source):
    """source: URL (str) or local path, for testing."""
    rec = {"file": key.split("/")[-1], "parse_error": ""}
    labels, protos = Counter(), Counter()
    fullchk = {c: [0, 0, 0, 0] for c in NUMERIC_FULL}
    rows = 0
    ts_min = ts_max = None
    sample_profile = None
    t0 = time.time()
    if isinstance(source, str) and source.startswith("http"):
        resp = requests.get(source, stream=True, timeout=(30, 600))
        resp.raise_for_status()
        resp.raw.decode_content = True
        raw = resp.raw
    else:
        raw = open(source, "rb")
    counter = LineCountingReader(raw)
    handle = io.BufferedReader(counter, buffer_size=1 << 22)
    try:
        header = handle.peek(1 << 16)[: 1 << 16].split(b"\n", 1)[0].decode("utf-8").rstrip("\r").split(",")
        rec["header"] = header
        reader = pd.read_csv(handle, dtype=str, keep_default_na=False, na_filter=False,
                             chunksize=CHUNK, on_bad_lines="error", encoding="utf-8")
        for i, ch in enumerate(reader):
            rows += len(ch)
            if i == 0:
                # ---- full-column profile on the first SAMPLE_ROWS rows (sample only) ----
                sm = ch.iloc[:SAMPLE_ROWS]
                sm = sm[sm["Label"] != "Label"]
                prof = {"rows": len(sm), "columns": {}}
                for c in sm.columns:
                    if c in ("Timestamp", "Label", "Flow ID", "Src IP", "Dst IP"):
                        continue
                    e, nn, inf, neg = numeric_issues(sm[c])
                    if e or nn or inf or neg:
                        prof["columns"][c] = {"empty": e, "non_numeric": nn, "inf": inf, "negative": neg}
                prof["exact_duplicate_rows"] = int(sm.duplicated().sum())
                feat = [c for c in sm.columns if c not in ("Timestamp", "Label", "Flow ID", "Src IP", "Dst IP", "Src Port")]
                prof["duplicate_rows_ignoring_timestamp"] = int(sm[feat + ["Label"]].duplicated().sum())
                fh = pd.util.hash_pandas_object(sm[feat], index=False)
                g = sm["Label"].groupby(fh.values).nunique()
                prof["feature_groups_with_conflicting_labels"] = int((g > 1).sum())
                prof["constant_columns"] = [c for c in sm.columns if sm[c].nunique() <= 1]
                sample_profile = prof
            labels.update(ch["Label"].value_counts().to_dict())
            body = ch[ch["Label"] != "Label"]                    # exclude repeated header lines
            protos.update(body["Protocol"].value_counts().to_dict())
            for c in NUMERIC_FULL:
                r = numeric_issues(body[c])
                fullchk[c] = [a + b for a, b in zip(fullchk[c], r)]
            ts = pd.to_datetime(body["Timestamp"], format="%d/%m/%Y %H:%M:%S", errors="coerce")
            if ts.notna().any():
                ts_min = min(filter(None, [ts_min, ts.min()]))
                ts_max = max(filter(None, [ts_max, ts.max()]))
            rec.setdefault("unparsed_timestamps", 0)
            rec["unparsed_timestamps"] += int(ts.isna().sum())
    except Exception as exc:
        rec["parse_error"] = f"{type(exc).__name__}: {exc}"
    while handle.read(1 << 22):
        pass
    rec["rows_parsed"] = rows
    rec["repeated_header_lines"] = int(labels.pop("Label", 0))
    rec["data_rows"] = rows - rec["repeated_header_lines"]
    rec["physical_lines"] = counter.physical_lines()
    rec["bytes_streamed"] = counter.nbytes
    rec["row_count_reconciles"] = counter.physical_lines() - 1 == rows
    rec["labels"] = dict(labels)
    rec["protocols"] = dict(protos)
    rec["full_file_numeric_checks"] = {c: dict(zip(["empty", "non_numeric", "inf", "negative"], v))
                                       for c, v in fullchk.items()}
    rec["timestamp_min"] = str(ts_min) if ts_min is not None else None
    rec["timestamp_max"] = str(ts_max) if ts_max is not None else None
    rec["sample_profile"] = sample_profile
    rec["seconds"] = round(time.time() - t0, 1)
    return rec


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out-dir", default="./h2_outputs")
    ap.add_argument("--local-test", nargs="*", help="TESTING ONLY: local CSV paths instead of S3")
    a = ap.parse_args()
    out = Path(a.out_dir); out.mkdir(parents=True, exist_ok=True)
    if a.local_test:
        files = [{"key": PREFIX + Path(p).name, "bytes": Path(p).stat().st_size, "etag": "", "src": p} for p in a.local_test]
    else:
        files = list_csvs()
        for f in files:
            f["src"] = BUCKET_URL + urllib.parse.quote(f["key"])
    results = []
    for f in files:
        rec = scan_file(f["key"], f["src"])
        rec.update({"listed_bytes": f["bytes"], "etag": f["etag"],
                    "complete_download": rec["bytes_streamed"] == f["bytes"]})
        results.append(rec)
        print(f"[scan] {rec['file']}: rows={rec['data_rows']:,} repeated_headers={rec['repeated_header_lines']} "
              f"bytes_ok={rec['complete_download']} reconcile={rec['row_count_reconciles']} "
              f"labels={len(rec['labels'])} {rec['seconds']}s err={rec['parse_error'] or '-'}", flush=True)
        (out / "scan_results.json").write_text(json.dumps(results, indent=1, default=str))
    print("done")


if __name__ == "__main__":
    main()


Writing h2_ids2018_scan.py


In [ ]:
# Step 3: integrity check of the scanner (locally tested copy md5 0724294fff7408b4b42747bea867c9e8), then the full streamed scan
import hashlib; print('scanner md5', hashlib.md5(open('h2_ids2018_scan.py','rb').read()).hexdigest())
!python -u h2_ids2018_scan.py --out-dir ./h2_outputs 2>&1 | tee h2_outputs/scan_log.txt

scanner md5 0724294fff7408b4b42747bea867c9e8
[scan] Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: rows=1,048,575 repeated_headers=0 bytes_ok=True reconcile=True labels=2 48.7s err=-
[scan] Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: rows=1,048,574 repeated_headers=1 bytes_ok=True reconcile=True labels=3 48.6s err=-
[scan] Friday-23-02-2018_TrafficForML_CICFlowMeter.csv: rows=1,048,575 repeated_headers=0 bytes_ok=True reconcile=True labels=4 59.5s err=-


In [ ]:
# Step 4: label summary from scan_results.json (exact label strings, repr shows hidden whitespace)
import json, pandas as pd
R = json.load(open('h2_outputs/scan_results.json'))
rows=[]
for r in R:
    for lab, n in r['labels'].items():
        rows.append({'file': r['file'][:22], 'label': repr(lab), 'n': n})
L = pd.DataFrame(rows)
print(L.pivot_table(index='label', columns='file', values='n', aggfunc='sum', fill_value=0).assign(TOTAL=lambda d: d.sum(axis=1)).sort_values('TOTAL', ascending=False)['TOTAL'].to_string())
print()
for r in R: print(r['file'][:22], {k: v for k, v in sorted(r['labels'].items(), key=lambda kv: -kv[1])})

In [ ]:
# Step 5: per-file rows, Excel-cap check, timestamp range, protocols, full-file numeric issues
EXCEL_MAX_DATA_LINES = 1_048_575          # Excel's 1,048,576-row limit minus the header line
for r in R:
    lines = r['physical_lines'] - 1
    fc = {k: {kk: vv for kk, vv in v.items() if vv} for k, v in r['full_file_numeric_checks'].items()}
    fc = {k: v for k, v in fc.items() if v}
    print(r['file'][:20], '| data lines', lines, '| at Excel cap', lines == EXCEL_MAX_DATA_LINES,
          '| ts', r['timestamp_min'][5:16], '->', r['timestamp_max'][5:16], '| unparsed ts', r['unparsed_timestamps'],
          '| proto', r['protocols'], '| issues', fc)

In [ ]:
# Step 6: sample profile (first 200,000 rows of each file - SAMPLE ONLY): duplicates, label conflicts, constant columns
for r in R:
    p = r['sample_profile']
    print(r['file'][:20], '| rows', p['rows'], '| exact dups', p['exact_duplicate_rows'], '| dups ignoring time', p['duplicate_rows_ignoring_timestamp'],
          '| conflicting groups', p['feature_groups_with_conflicting_labels'], '| constant', len(p['constant_columns']))
const = [set(r['sample_profile']['constant_columns']) for r in R]
print('\nconstant in ALL 10 samples:', sorted(set.intersection(*const)))
nonnum = sorted({c for r in R for c, v in r['sample_profile']['columns'].items() if v['non_numeric']})
print('columns with non-numeric text in samples:', nonnum)

In [ ]:
# Step 7: what are the non-numeric 'Flow Byts/s' values, and what do Protocol 0 / conflicting-label rows look like? (first 40 MB of one file)
import io
b = requests.get(url('Processed Traffic Data for ML Algorithms/Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv'), headers={'Range':'bytes=0-41943039'}, timeout=120).content
s = pd.read_csv(io.BytesIO(b[:b.rfind(b'\n')]), dtype=str, keep_default_na=False)
s = s[s.Label != 'Label']
v = s['Flow Byts/s']; bad = v[pd.to_numeric(v.where(~v.str.lower().isin(['inf','infinity'])), errors='coerce').isna()]
print('non-numeric Flow Byts/s tokens:', bad.value_counts().to_dict())
print('Protocol 0 rows by label:', s[s.Protocol=='0'].Label.value_counts().to_dict(), '| Dst Port of proto-0 rows:', s[s.Protocol=='0']['Dst Port'].value_counts().head(3).to_dict())
feat = [c for c in s.columns if c not in ('Timestamp','Label')]
g = s.groupby(pd.util.hash_pandas_object(s[feat], index=False).values)
conf = g.Label.nunique(); conf_keys = conf[conf>1].index
cs = s[pd.util.hash_pandas_object(s[feat], index=False).isin(conf_keys).values]
print('rows in conflicting groups:', len(cs), 'by label:', cs.Label.value_counts().to_dict())
print('example conflicting group (Dst Port, Protocol, Flow Duration, Tot Fwd Pkts, Tot Bwd Pkts, Label):')
print(cs.sort_values(feat[:5])[['Dst Port','Protocol','Flow Duration','Tot Fwd Pkts','Tot Bwd Pkts','Label']].head(6).to_string(index=False))

In [ ]:
%%writefile h2_make_report.py
#!/usr/bin/env python3
"""
SentinelAI H-2 - build the CSE-CIC-IDS2018 backup-dataset deliverables from the
evidence written by h2_ids2018_scan.py (and the H-1 outputs for the AIT comparison).
Every number is read from those files; nothing is typed in by hand.

Outputs (in --out-dir):
  ids2018_file_structure.csv      per-file size, columns, rows, labels, timestamps
  ids2018_label_counts_by_file.csv exact label counts per file with proposed binary label
  ids2018_label_mapping.csv/.json  proposed label -> Normal(0)/Suspicious(1) mapping
  ait_vs_ids2018_comparison.csv    side-by-side comparison table
  H2_CSE-CIC-IDS2018_Verification_Report.md
  documentation_paragraph.md       short paragraph for the project documentation

Run: python h2_make_report.py --out-dir ./h2_outputs --h1-dir ./h1_outputs
"""
from __future__ import annotations

import argparse
import json
import re
from pathlib import Path

import pandas as pd

EXCEL_MAX_DATA_LINES = 1_048_575   # Excel worksheet limit (1,048,576 rows) minus the header line

# Proposed mapping. CIC publishes no benign/malicious label file (unlike AIT's label_info.txt),
# so each observed label is tied to an attack scenario from the official UNB page.
# Only the exact string "Benign" is Normal; any value not listed here stays UNMAPPED.
PROPOSED = {
    "Benign": ("Normal traffic", 0),
    "FTP-BruteForce": ("Brute-force (FTP, Patator) - 14-02", 1),
    "SSH-Bruteforce": ("Brute-force (SSH, Patator) - 14-02", 1),
    "DoS attacks-GoldenEye": ("DoS (GoldenEye) - 15-02", 1),
    "DoS attacks-Slowloris": ("DoS (Slowloris) - 15-02", 1),
    "DoS attacks-SlowHTTPTest": ("DoS (SlowHTTPTest) - 16-02", 1),
    "DoS attacks-Hulk": ("DoS (Hulk) - 16-02", 1),
    "DDoS attacks-LOIC-HTTP": ("DDoS (LOIC-HTTP) - 20-02", 1),
    "DDOS attack-LOIC-UDP": ("DDoS (LOIC-UDP) - 20-02 and 21-02", 1),
    "DDOS attack-HOIC": ("DDoS (HOIC) - 21-02", 1),
    "Brute Force -Web": ("Web attack: brute force - 22-02, 23-02", 1),
    "Brute Force -XSS": ("Web attack: XSS - 22-02, 23-02", 1),
    "SQL Injection": ("Web attack: SQL injection - 22-02, 23-02", 1),
    "Infilteration": ("Infiltration from inside (spelling as in the data) - 28-02, 01-03", 1),
    "Bot": ("Botnet (Zeus, Ares) - 02-03", 1),
}


def md(df: pd.DataFrame) -> str:
    cols = [str(c) for c in df.columns]
    out = ["| " + " | ".join(cols) + " |", "|" + "|".join("---" for _ in cols) + "|"]
    for _, r in df.iterrows():
        cells = []
        for v in r.tolist():
            if isinstance(v, bool):
                cells.append("yes" if v else "no")
            elif isinstance(v, int):
                cells.append(f"{v:,}")
            elif isinstance(v, float):
                cells.append(f"{v:,.2f}")
            else:
                cells.append(str(v).replace("|", "\\|"))
        out.append("| " + " | ".join(cells) + " |")
    return "\n".join(out)


def day_of(fname: str) -> str:
    return fname.split("_")[0]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out-dir", default="./h2_outputs")
    ap.add_argument("--h1-dir", default="./h1_outputs")
    a = ap.parse_args()
    O, H1 = Path(a.out_dir), Path(a.h1_dir)

    R = json.loads((O / "scan_results.json").read_text())
    lst = pd.read_csv(O / "s3_bucket_listing.csv")
    headers = json.loads((O / "csv_headers.json").read_text())
    log = (O / "scan_log.txt").read_text() if (O / "scan_log.txt").exists() else ""

    # ------------------------------------------------------------------ bucket
    lst["top"] = lst["key"].str.split("/").str[0]
    raw = lst[lst["top"] == "Original Network Traffic and Log data"]
    proc = lst[lst["top"] == "Processed Traffic Data for ML Algorithms"]
    gib = lambda b: b / 2**30
    raw_pcap = raw[raw["key"].str.contains("pcap", case=False)]
    raw_logs = raw[raw["key"].str.contains("logs", case=False)]

    # ------------------------------------------------------------------ per file
    base_hdr = max(headers.values(), key=lambda h: list(headers.values()).count(h))
    fs = []
    lc = []
    for r in R:
        f = r["file"]
        hdr = r["header"]
        data_lines = r["physical_lines"] - 1
        labels = r["labels"]
        attacks = {k: v for k, v in labels.items() if k != "Benign"}
        fs.append({
            "file": f, "day": day_of(f), "MB": round(r["listed_bytes"] / 1e6, 1),
            "columns": len(hdr),
            "extra_columns_vs_common_header": ", ".join(c for c in hdr if c not in base_hdr) or "-",
            "data_rows": r["data_rows"], "repeated_header_lines": r["repeated_header_lines"],
            "lines_at_excel_limit": data_lines == EXCEL_MAX_DATA_LINES,
            "benign": labels.get("Benign", 0), "attack_rows": sum(attacks.values()),
            "attack_pct": round(100 * sum(attacks.values()) / r["data_rows"], 2),
            "attack_labels": "; ".join(f"{k} ({v:,})" for k, v in sorted(attacks.items(), key=lambda kv: -kv[1])),
            "timestamp_min": r["timestamp_min"], "timestamp_max": r["timestamp_max"],
            "protocol_counts": json.dumps(r["protocols"]),
            "bytes_streamed_equals_listed": r["complete_download"], "row_count_reconciles": r["row_count_reconciles"],
            "parse_error": r["parse_error"] or "none",
        })
        for lab, n in labels.items():
            fam, y = PROPOSED.get(lab, ("UNMAPPED - not a documented label", ""))
            lc.append({"file": f, "day": day_of(f), "label": lab, "label_repr": repr(lab), "flows": n,
                       "attack_scenario": fam, "binary_label": y})
    fs = pd.DataFrame(fs)
    lc = pd.DataFrame(lc)
    fs.to_csv(O / "ids2018_file_structure.csv", index=False)
    lc.to_csv(O / "ids2018_label_counts_by_file.csv", index=False)

    tot = lc.groupby("label").agg(flows=("flows", "sum"), files=("file", "nunique"),
                                  days=("day", lambda d: ", ".join(sorted(set(d))))).reset_index()
    tot["attack_scenario"] = tot["label"].map(lambda l: PROPOSED.get(l, ("UNMAPPED", ""))[0])
    tot["binary_label"] = tot["label"].map(lambda l: PROPOSED.get(l, ("", ""))[1])
    tot = tot.sort_values("flows", ascending=False)
    tot.to_csv(O / "ids2018_label_mapping.csv", index=False)
    unmapped = tot[tot["binary_label"] == ""]
    total_rows = int(fs["data_rows"].sum())
    normal = int(tot.loc[tot["binary_label"] == 0, "flows"].sum())
    susp = int(tot.loc[tot["binary_label"] == 1, "flows"].sum())
    (O / "ids2018_label_mapping.json").write_text(json.dumps({
        "dataset": "CSE-CIC-IDS2018, Processed Traffic Data for ML Algorithms (10 CSV files)",
        "status": "PROPOSED by H-2 (CIC publishes no benign/malicious label file); to be confirmed in H-3",
        "encoding": {"Normal": 0, "Suspicious": 1},
        "rules": ["Exact, case-sensitive match on the Label column.",
                  "Only 'Benign' maps to 0. Each documented attack label maps to 1.",
                  "Rows whose Label is the literal text 'Label' are repeated header lines and are dropped, not labelled.",
                  "Any other value is UNMAPPED and is never treated as benign."],
        "mapping": {k: {"binary": v[1], "scenario": v[0]} for k, v in PROPOSED.items()},
        "observed_unmapped_labels": unmapped["label"].tolist(),
    }, indent=2))

    # ------------------------------------------------------------------ quality totals
    def qsum(col, kind):
        return sum(r["full_file_numeric_checks"][col][kind] for r in R)
    nan_text = qsum("Flow Byts/s", "non_numeric")
    inf_b = qsum("Flow Byts/s", "inf")
    inf_p = qsum("Flow Pkts/s", "inf")
    neg_dur = qsum("Flow Duration", "negative")
    neg_win = qsum("Init Fwd Win Byts", "negative")
    rep_hdr = int(fs["repeated_header_lines"].sum())
    at_cap = fs[fs["lines_at_excel_limit"]]
    proto = pd.Series(dtype="int64")
    for r in R:
        proto = proto.add(pd.Series(r["protocols"]), fill_value=0)
    proto = proto.astype(int)
    hours_ok = all(r["timestamp_min"] and r["timestamp_max"] for r in R)
    ts_hours = sorted({int(r["timestamp_min"][11:13]) for r in R} | {int(r["timestamp_max"][11:13]) for r in R})
    max_hour = max(int(r["timestamp_max"][11:13]) for r in R)
    out_of_day = [r["file"] for r in R if r["timestamp_min"][:10] != r["timestamp_max"][:10]]
    s_rows = sum(r["sample_profile"]["rows"] for r in R)
    s_dups = sum(r["sample_profile"]["exact_duplicate_rows"] for r in R)
    s_dups_nt = sum(r["sample_profile"]["duplicate_rows_ignoring_timestamp"] for r in R)
    s_conf = {r["file"]: r["sample_profile"]["feature_groups_with_conflicting_labels"] for r in R
              if r["sample_profile"]["feature_groups_with_conflicting_labels"]}
    const_all = sorted(set.intersection(*[set(r["sample_profile"]["constant_columns"]) for r in R]))
    loic_udp_tue = "DDOS attack-LOIC-UDP" in next(r for r in R if r["file"].startswith("Thuesday"))["labels"]
    ip_files = [r["file"] for r in R if "Src IP" in r["header"]]

    # ------------------------------------------------------------------ AIT (from H-1 outputs)
    ait = pd.read_csv(H1 / "ait_network_attack_counts.csv")
    ait_tot = ait[ait["network"] == "TOTAL"].iloc[0]
    ait_q = pd.read_csv(H1 / "csv_file_quality_report.csv")
    ait_cols = sorted(ait_q["n_columns"].unique().tolist(), reverse=True)
    ait_arc = pd.read_csv(H1 / "archive_checksums.csv")
    ait_mb = round(ait_arc["actual_bytes"].sum() / 1e6, 1)
    ait_map = json.loads((H1 / "label_mapping.json").read_text())
    ait_ben = len(ait_map["TCP"]["benign_0"]) + len(ait_map["UDP"]["benign_0"])
    ait_mal = len(ait_map["TCP"]["malicious_1"]) + len(ait_map["UDP"]["malicious_1"])
    ait_unm = int(ait_tot["unmapped_flows"])
    ait_nc = 100 * ait_tot["tcp_nocomplete_suspicious_flows"] / ait_tot["tcp_nocomplete_total_flows"]
    scan_min = sum(r["seconds"] for r in R) / 60
    ws_labels = [l for l in tot["label"] if l != l.strip()]
    norm = {}
    for l in tot["label"]:
        norm.setdefault(re.sub(r"[^a-z]", "", l.lower()), []).append(l)
    near_dups = [v for v in norm.values() if len(v) > 1]
    heartbleed = [l for l in tot["label"] if "heart" in l.lower()]
    dup_ratio = sorted(((r["sample_profile"]["exact_duplicate_rows"] / max(r["sample_profile"]["rows"], 1), day_of(r["file"]))
                        for r in R), reverse=True)[:2]

    comp = pd.DataFrame([
        ["Publisher / source", "AIT Austrian Institute of Technology, Zenodo record 13168643 (2024)",
         "CSE and Canadian Institute for Cybersecurity (UNB), public AWS bucket s3://cse-cic-ids2018 (2018)"],
        ["Environment", "8 separate simulated small-company networks, 4-6 days each",
         "1 emulated organisation (420 machines, 30 servers, 50 attackers), 10 capture days"],
        ["Files used", f"8 ZIPs, 24 CSVs (Tstat), {ait_mb} MB compressed",
         f"{len(fs)} CSVs (CICFlowMeter-V3), {gib(proc['bytes'].sum()):.2f} GiB uncompressed; raw PCAP/logs {gib(raw['bytes'].sum()):.1f} GiB"],
        ["Flows", f"{int(ait_tot['total_flows']):,}", f"{total_rows:,}"],
        ["Suspicious share", f"{float(ait_tot['suspicious_pct']):.2f}%", f"{100 * susp / total_rows:.2f}%"],
        ["Columns", f"{', '.join(map(str, ait_cols))} (TCP complete / TCP non-complete / UDP)",
         f"{len(base_hdr)} on {len(fs) - len(ip_files)} days; {max(len(h) for h in headers.values())} on {len(ip_files)} day"],
        ["Host identifiers (IP addresses)", "Yes, in every file (client and server IP, host roles)",
         f"Only in {len(ip_files)} of {len(fs)} files ({', '.join(day_of(f) for f in ip_files)})"],
        ["Timestamps", "Epoch milliseconds plus full date-time", "dd/mm/yyyy hh:mm:ss, hours only 01-12 (no AM/PM)"],
        ["Labels", f"Official label_info.txt: {ait_ben} benign + {ait_mal} malicious labels, protocol-aware",
         f"{len(tot)} values in a single Label column; no official benign/malicious file"],
        ["Attack types", "One multi-step attack per network (scans, web attack, web shell, commands, DNS exfiltration)",
         "Brute force, DoS, DDoS, web attacks, infiltration, botnet - one or two types per day"],
        ["Binary mapping", f"Exact and official; {ait_unm:,} flows unmapped", f"Straightforward (Benign vs rest); {len(unmapped)} unmapped values"],
        ["Main quality issues", f"{ait_unm:,} empty labels; placeholder text; {ait_nc:.0f}% attacks in TCP non-complete",
         "Inf/NaN text, negative values, repeated headers, row-limit files, 12-hour clock, label noise"],
        ["Split by network (current plan)", "Yes - 8 networks", "No - only one network; split by day = split by attack type"],
        ["Per-host 5-minute windows (H-4)", "Possible", f"Only possible on {len(ip_files)} day; not on the others"],
        ["Matching alert data for agents", "AIT Alert Data Set from the same attacks", "None provided by CIC"],
        ["Processing in Colab", "Easy (a few minutes)", f"Feasible by streaming in chunks ({scan_min:.0f} minutes for all {len(fs)} files)"],
    ], columns=["Aspect", "AIT Netflow (primary)", "CSE-CIC-IDS2018 (backup)"])
    comp.to_csv(O / "ait_vs_ids2018_comparison.csv", index=False)

    # ------------------------------------------------------------------ report
    L = []
    w = L.append
    w("# H-2 — CSE-CIC-IDS2018 Backup Dataset Verification")
    w("")
    w("*SentinelAI · Data & Features (Habiba) · generated by \x60h2_make_report.py\x60 from the evidence files in "
      "\x60h2_outputs/\x60. Verified = checked by our own code on the real files; Documented = stated by an official or "
      "published source and not re-checked by us.*")
    w("")
    w("**Decision: Suitable with conditions** — as a backup for the Random Forest detection part (objective O1), "
      "not as a drop-in replacement for the whole SentinelAI pipeline. See section 9.")
    w("")
    w("## 1. Objective and link to H-1")
    w("")
    w("Tracker task H-2: check the CSE-CIC-IDS2018 backup (size, format, labels). Done when there is a short note on how "
      "to switch if AIT fails the gate. H-1 gave AIT a conditional GO, so this backup is not needed now. H-2 makes sure "
      "the team knows, with evidence, what switching would involve and cost.")
    w("")
    w("## 2. Official source (Documented)")
    w("")
    w("- Official page: Canadian Institute for Cybersecurity, University of New Brunswick — "
      "https://www.unb.ca/cic/datasets/ids-2018.html (collaboration between the Communications Security Establishment "
      "and CIC).")
    w("- Official download: public AWS bucket \x60s3://cse-cic-ids2018/\x60 (region ca-central-1), AWS Open Data registry "
      "https://registry.opendata.aws/cse-cic-ids2018/. No account is needed. The page asks users to cite the dataset and "
      "link the registry page.")
    w("- Paper: I. Sharafaldin, A. H. Lashkari, A. A. Ghorbani, *Toward Generating a New Intrusion Detection Dataset and "
      "Intrusion Traffic Characterization*, ICISSP 2018.")
    w("- Purpose: a benchmark for network intrusion detection, built from benign user profiles and attack profiles. "
      "Seven attack scenarios (brute force, Heartbleed, botnet, DoS, DDoS, web attacks, infiltration) were run by 50 attacker "
      "machines against an organisation of 5 departments, 420 machines and 30 servers. Flow features were extracted "
      "with CICFlowMeter-V3.")
    w("")
    w("## 3. Bucket structure and size (Verified)")
    w("")
    w(f"The bucket was listed anonymously from Colab (\x60s3_bucket_listing.csv\x60): **{len(lst)} objects, "
      f"{gib(lst['bytes'].sum()):.2f} GiB in total**.")
    w("")
    w(md(pd.DataFrame([
        ["Original Network Traffic and Log data", len(raw), f"{gib(raw['bytes'].sum()):.2f} GiB",
         f"Per day: one PCAP archive ({gib(raw_pcap['bytes'].min()):.1f}–{gib(raw_pcap['bytes'].max()):.1f} GiB) "
         f"and one logs.zip ({gib(raw_logs['bytes'].min()):.2f}–{gib(raw_logs['bytes'].max()):.2f} GiB)"],
        ["Processed Traffic Data for ML Algorithms", len(proc), f"{gib(proc['bytes'].sum()):.2f} GiB",
         f"{len(fs)} CSV files, one per capture day (labelled flows)"]],
        columns=["Folder", "Objects (incl. folder markers)", "Size", "Contents"])))
    w("")
    w("Only the processed CSVs are practical for SentinelAI in Colab. The raw PCAPs (hundreds of GiB) would only be "
      "needed to regenerate flows with IP addresses or corrected labels.")
    w("")
    w("## 4. File structure and format (Verified)")
    w("")
    w(f"All {len(fs)} CSVs were streamed in full from the bucket. The number of bytes read equals the listed size for every "
      f"file{'' if fs['bytes_streamed_equals_listed'].all() else ' EXCEPT those marked no'}, there were "
      f"{'no parse errors' if (fs['parse_error'] == 'none').all() else 'parse errors (see table)'}, and pandas row counts "
      f"equal the physical line counts.")
    w("")
    show = fs[["day", "MB", "columns", "data_rows", "repeated_header_lines", "lines_at_excel_limit", "benign",
               "attack_rows", "attack_pct"]].copy()
    show = pd.concat([show, pd.DataFrame([{"day": "TOTAL", "MB": round(fs["MB"].sum(), 1), "columns": "",
                                           "data_rows": total_rows, "repeated_header_lines": rep_hdr,
                                           "lines_at_excel_limit": "", "benign": int(lc.loc[lc.label == "Benign", "flows"].sum()),
                                           "attack_rows": susp, "attack_pct": round(100 * susp / total_rows, 2)}])])
    w(md(show))
    w("")
    w(f"- **Format:** comma-separated text with one header row; one row is one bidirectional flow. "
      f"{len(fs) - len(ip_files)} files share the same {len(base_hdr)}-column header. "
      f"{', '.join(ip_files)} has {max(len(h) for h in headers.values())} columns: the extra columns are "
      f"{', '.join(c for c in headers[ip_files[0]] if c not in base_hdr)}. Shared columns are in the same order.")
    w(f"- **Organisation:** by day, not by attack. Each day file mixes benign traffic with that day's one or two attacks.")
    w(f"- **Size in practice:** the largest file is {fs['MB'].max():,.0f} MB with {fs['data_rows'].max():,} rows. "
      "It fits Colab only when read in chunks.")
    w("")
    w(f"### Columns (the common {len(base_hdr)}-column header)")
    w("")
    w("Identifiers and time: \x60Dst Port\x60, \x60Protocol\x60, \x60Timestamp\x60. Label: \x60Label\x60. The other "
      f"{len(base_hdr) - 4} columns are numeric CICFlowMeter statistics, grouped as follows (meanings from the official page):")
    w("")
    groups = [
        ("Duration and counts", ["Flow Duration", "Tot Fwd Pkts", "Tot Bwd Pkts", "TotLen Fwd Pkts", "TotLen Bwd Pkts"],
         "How long the flow lasted and how many packets/bytes went each way"),
        ("Packet length", [c for c in base_hdr if "Pkt Len" in c], "Size statistics of packets, per direction and overall"),
        ("Rates", ["Flow Byts/s", "Flow Pkts/s", "Fwd Pkts/s", "Bwd Pkts/s"], "Bytes and packets per second"),
        ("Inter-arrival times", [c for c in base_hdr if "IAT" in c], "Time gaps between packets"),
        ("TCP flags", [c for c in base_hdr if "Flag" in c or "Flags" in c], "Counts of FIN, SYN, RST, PSH, ACK, URG, CWE, ECE"),
        ("Headers, segments, windows", [c for c in base_hdr if "Header" in c or "Seg" in c or "Win" in c or c == "Fwd Act Data Pkts"],
         "Header lengths, segment sizes, initial TCP window sizes"),
        ("Bulk and sub-flow", [c for c in base_hdr if "Blk" in c or "b Avg" in c or "Subflow" in c], "Bulk-transfer and sub-flow statistics"),
        ("Active / idle", [c for c in base_hdr if c.startswith("Active") or c.startswith("Idle")], "How long the flow was active or idle"),
        ("Other", ["Down/Up Ratio", "Pkt Size Avg"], "Download/upload ratio, average packet size"),
    ]
    used = set()
    rows = []
    for g, cols, meaning in groups:
        cols = [c for c in cols if c in base_hdr and c not in used]
        used.update(cols)
        rows.append([g, len(cols), ", ".join(f"\x60{c}\x60" for c in cols), meaning])
    rest = [c for c in base_hdr if c not in used and c not in ("Dst Port", "Protocol", "Timestamp", "Label")]
    if rest:
        rows.append(["Ungrouped", len(rest), ", ".join(f"\x60{c}\x60" for c in rest), "-"])
    w(md(pd.DataFrame(rows, columns=["Group", "Columns", "Names (exact)", "Meaning"])))
    w("")
    w("## 5. Labels (Verified) and proposed binary mapping")
    w("")
    w(f"The label column is \x60Label\x60 in every file. The full scan found **{len(tot)} distinct values** in "
      f"{total_rows:,} flows. Strings were compared exactly; "
      f"{'none contain leading or trailing whitespace' if not ws_labels else 'some contain stray whitespace: ' + repr(ws_labels)}. "
      f"{rep_hdr} further rows contain the literal text \x60Label\x60: these are repeated header lines inside the data "
      "and are not flows.")
    w("")
    w("CIC publishes no benign/malicious list (AIT has \x60label_info.txt\x60), so the mapping below is **proposed by H-2**. "
      "Each attack label is tied to a scenario on the official page. Only the exact string \x60Benign\x60 maps to 0; "
      "any unknown value stays unmapped.")
    w("")
    t = tot[["label", "attack_scenario", "binary_label", "flows", "days"]].copy()
    t["label"] = t["label"].map(lambda s: f"\x60{s}\x60")
    t["share %"] = (100 * tot["flows"] / total_rows).round(3)
    w(md(t))
    w("")
    w(f"Result: Normal (0) = **{normal:,}**, Suspicious (1) = **{susp:,}** ({100 * susp / total_rows:.2f}%), "
      f"unmapped = **{int(unmapped['flows'].sum()):,}**.")
    w("")
    w("**Label consistency (Verified):**")
    w("")
    multi = tot[tot["files"] > 1]
    w(f"- Labels used on more than one day: "
      f"{', '.join(f'{chr(96)}{l}{chr(96)} ({d})' for l, d in zip(multi['label'], multi['days'])) or 'none'}. "
      + ("No two label values differ only in case, spacing or punctuation, so each attack has one spelling across all files."
         if not near_dups else f"Near-duplicate spellings found: {near_dups}."))
    w("- Naming is not uniform: \x60DDOS attack-HOIC\x60 / \x60DDOS attack-LOIC-UDP\x60 versus \x60DDoS attacks-LOIC-HTTP\x60; "
      "\x60Infilteration\x60 is misspelt; \x60Brute Force -Web\x60 has an odd space. This is harmless for an exact mapping "
      "but must be copied exactly.")
    w(f"- **Missing attack:** the official schedule lists DDoS LOIC-UDP on 20-02, but the 20-02 file contains "
      f"{'that label' if loic_udp_tue else 'no LOIC-UDP label'}. LOIC-UDP appears in: "
      f"{', '.join(tot.loc[tot.label == 'DDOS attack-LOIC-UDP', 'days']) or 'no file'} "
      f"({int(tot.loc[tot.label == 'DDOS attack-LOIC-UDP', 'flows'].sum()):,} flows). Heartbleed is described on the "
      + ("official page but has no label in any file." if not heartbleed else f"official page; labels found: {heartbleed}."))
    w(f"- **Label noise:** in the first 200,000 rows of each file, groups of rows with identical features but different labels "
      f"were found in {', '.join(f'{day_of(k)} ({v:,} groups)' for k, v in s_conf.items())}. A follow-up check of the first "
      "40 MB of the 01-03 file showed these groups mix \x60Benign\x60 and \x60Infilteration\x60. Published work also reports "
      "labelling errors in this dataset (Liu et al., 2022).")
    w("")
    w("## 6. Data quality (Verified)")
    w("")
    w("Full-file checks cover every row for the listed columns; sample checks cover the first 200,000 rows of each file "
      f"({s_rows:,} rows in total) and are marked as such.")
    w("")
    w(md(pd.DataFrame([
        ["Literal 'NaN' text in Flow Byts/s", "full", nan_text],
        ["'Infinity' in Flow Byts/s", "full", inf_b],
        ["'Infinity' in Flow Pkts/s", "full", inf_p],
        ["Negative Flow Duration", "full", neg_dur],
        ["Negative Init Fwd Win Byts (-1 = not observed)", "full", neg_win],
        ["Repeated header lines inside data", "full", rep_hdr],
        ["Exact duplicate rows", "sample", s_dups],
        ["Duplicate rows if Timestamp is ignored", "sample", s_dups_nt],
        ["Columns constant in every sample", "sample", len(const_all)],
    ], columns=["Check", "Scope", "Count"])))
    w("")
    w(f"- **Row-limit files:** {len(at_cap)} of {len(fs)} files have exactly {EXCEL_MAX_DATA_LINES:,} data lines "
      f"({', '.join(at_cap['day'])}). This is Excel's maximum sheet size minus the header row. It strongly suggests these "
      "files passed through a spreadsheet and **may be truncated**. We could not confirm it from the times (next point); "
      "confirming it would need the raw PCAPs.")
    w(f"- **12-hour timestamps:** the latest time in any file is {max_hour}:59; no hour from 13 to 23 occurs, "
      "and there is no AM/PM marker. So morning and afternoon flows cannot be told apart, and the time order within a day is ambiguous. "
      + (f"Some timestamps also fall outside the capture day ({', '.join(day_of(f) for f in out_of_day)})." if out_of_day else ""))
    w(f"- **Protocols:** {', '.join(f'{k}: {v:,}' for k, v in proto.items())} flows (6 = TCP, 17 = UDP, 0 = no transport "
      "protocol; in a checked sample all protocol-0 flows had destination port 0).")
    w(f"- **Constant columns** (no information in any sample): {', '.join(f'{chr(96)}{c}{chr(96)}' for c in const_all)}.")
    w(f"- **Duplicates** are very uneven by day. In the samples, {dup_ratio[0][0]:.0%} of rows in {dup_ratio[0][1]} and "
      f"{dup_ratio[1][0]:.0%} in {dup_ratio[1][1]} are exact duplicates. Duplicates that end up on both sides of the "
      "train/test split inflate the scores.")
    w("")
    w("## 7. Comparison with AIT (from H-1 evidence and this scan)")
    w("")
    w(md(comp))
    w("")
    w("## 8. Fit with SentinelAI and the Random Forest")
    w("")
    w("- **Random Forest: yes.** The 70+ numeric per-flow features are the kind of input a Random Forest handles well, "
      "and binary labels are easy to derive. After cleaning (Inf/NaN, constant columns, repeated headers) a flow-level "
      "model can be trained in Colab.")
    w("- **Planned host-window features (H-4): mostly no.** Connections, distinct ports and distinct hosts per host in a "
      f"past-only 5-minute window need a source IP and a reliable time order. Only {len(ip_files)} of {len(fs)} files have "
      "IPs, and the 12-hour clock makes time order ambiguous.")
    w("- **Fair split (H-6): changes.** There is one network, not eight. Splitting by day means each test day has attack "
      "types the model never saw, which measures something different. A random split leaks near-duplicate flows.")
    w("- **Agents:** CIC provides no IDS alert set like the AIT Alert Data Set. The Investigation Agent would need "
      "synthetic or self-generated alerts. ATT&CK/Kill-Chain mappings would cover different techniques (brute force, DoS, "
      "botnet) than the AIT attack chain.")
    w("")
    w("## 9. Backup decision")
    w("")
    w("1. **Can it serve as a backup?** Yes, for the Random Forest detection part. It is official, public, labelled, "
      "and processable in Colab.")
    w("2. **Advantages over AIT:** many more flows and attack types (brute force, DoS/DDoS, web attacks, botnet, "
      "infiltration); widely used in published work, so results are easy to compare; a large attack share in several files.")
    w("3. **Disadvantages:** one network only (no split by network); IPs in only one file; 12-hour timestamps; files "
      "that may be truncated; Inf/NaN and negative values; repeated headers; duplicates; documented and observed label "
      "noise (especially infiltration); no matching alert data; no official binary label file.")
    w("4. **Modifications needed:** drop repeated header lines; replace or remove NaN/Infinity rows; treat -1 window values "
      "as 'not observed'; drop constant columns; remove exact duplicates before splitting; apply the exact label mapping "
      "above; consider excluding \x60Infilteration\x60 or reporting results with and without it.")
    w("5. **Methodology changes if we switch:** feature module without per-host windows (or limited to the 20-02 file); "
      "a new split strategy (by day/attack type, or a deduplicated stratified split, stated clearly); retargeted rules "
      "(M-2) and ATT&CK subset (O-1); alerts for the agents generated from our own detections instead of AIT-ADS; "
      "updated proposal slides 8–11.")
    w("")
    w("**Recommendation: Suitable with conditions.** Keep AIT as the primary dataset (H-1: conditional GO). Use CSE-CIC-IDS2018 "
      "(a) as the backup for the flow-level Random Forest if AIT fails, and (b) optionally as a second, out-of-distribution "
      "test set, as the proposal already allows. It is not a full replacement for the multi-agent part without extra work.")
    w("")
    w("### How to switch if AIT fails the gate (tracker completion note)")
    w("")
    w("1. Rerun \x60h2_ids2018_scan.py\x60 to re-verify the 10 CSVs, and lock the label mapping (\x60ids2018_label_mapping.json\x60) in H-3.")
    w("2. Build a cleaning step: drop rows with Label = 'Label'; handle NaN/Infinity; drop constant columns; deduplicate.")
    w("3. Agree a new feature contract (ALL-1) based on per-flow CICFlowMeter features only, without IPs, ports used as "
      "identities, or timestamps.")
    w("4. Choose and document the split (by day / attack type, or a deduplicated stratified split) and update H-6.")
    w("5. Tell Malak (M-2/M-3), Reem (R-1/R-4) and Omar (O-1/O-3) which attack types and alert sources change.")
    w("")
    w("## 10. Limitations of this verification")
    w("")
    w("- Duplicate, conflict and constant-column checks used the first 200,000 rows of each file, not every row.")
    w("- Possible truncation was inferred from row counts; it was not confirmed against the raw PCAPs.")
    w("- S3 ETags were recorded, but CIC publishes no checksums for these files, so integrity can only be confirmed as "
      "'complete download of the published bytes'.")
    w("- The binary mapping is proposed, not official. H-3 owns the final labelling rule.")
    w("- No model was trained; suitability for the Random Forest is judged from the features and labels, not from results.")
    w("")
    w("## 11. Paragraph for the project documentation")
    w("")
    para = (f"As a backup to the AIT Netflow Data Set we verified CSE-CIC-IDS2018 (CSE and the Canadian Institute for "
            f"Cybersecurity, 2018), using its official public AWS release. The ten processed CICFlowMeter CSV files "
            f"({gib(proc['bytes'].sum()):.2f} GiB, {total_rows:,} labelled flows) were streamed in full. They contain "
            f"{len(tot)} label values: 'Benign' and {len(tot) - 1} attack labels covering brute force, DoS, DDoS, web "
            f"attacks, infiltration and botnet traffic, giving {100 * susp / total_rows:.1f}% suspicious flows under our "
            f"binary mapping. The data supports a flow-level Random Forest, but it comes from a single network. IP addresses "
            f"are present on only one of ten days, and timestamps use a 12-hour clock without AM/PM, so our per-host time-window "
            f"features and network-wise split cannot be reproduced. The files also need cleaning for Infinity/NaN values, "
            f"repeated header lines, duplicates and known label noise. We therefore judge CSE-CIC-IDS2018 suitable, with "
            f"conditions, as a backup for the detection model and as an optional second test set, while AIT remains the "
            f"primary dataset.")
    w(para)
    (O / "documentation_paragraph.md").write_text(para + "\n")
    w("")
    w("## Evidence and sources")
    w("")
    w("- Evidence files: \x60s3_bucket_listing.csv\x60, \x60csv_headers.json\x60, \x60scan_results.json\x60, \x60scan_log.txt\x60, "
      "\x60ids2018_file_structure.csv\x60, \x60ids2018_label_counts_by_file.csv\x60, \x60ids2018_label_mapping.csv/.json\x60, "
      "\x60ait_vs_ids2018_comparison.csv\x60; scripts \x60h2_ids2018_scan.py\x60, \x60h2_make_report.py\x60.")
    w("- CIC dataset page: https://www.unb.ca/cic/datasets/ids-2018.html")
    w("- AWS Open Data registry: https://registry.opendata.aws/cse-cic-ids2018/")
    w("- Sharafaldin, Lashkari, Ghorbani, ICISSP 2018 (dataset paper).")
    w("- Liu, Engelen, Lynar, Essam, Joosen, *Error Prevalence in NIDS datasets: A Case Study on CIC-IDS-2017 and "
      "CSE-CIC-IDS-2018*, IEEE CNS 2022, doi:10.1109/CNS56114.2022.9947235.")
    w("- Leevy and Khoshgoftaar, *A survey and analysis of intrusion detection models based on CSE-CIC-IDS2018 Big Data*, "
      "Journal of Big Data 7:104 (2020), doi:10.1186/s40537-020-00382-x.")
    w("- H-1 outputs (AIT numbers): \x60h1_outputs/ait_network_attack_counts.csv\x60, \x60csv_file_quality_report.csv\x60.")
    w("")
    (O / "H2_CSE-CIC-IDS2018_Verification_Report.md").write_text("\n".join(L), encoding="utf-8")
    print("written; flows", total_rows, "normal", normal, "suspicious", susp, "unmapped", int(unmapped["flows"].sum()),
          "| at excel cap", len(at_cap), "| ip files", ip_files, "| hours", ts_hours)


if __name__ == "__main__":
    main()


In [ ]:
# Step 8: integrity check of the report generator (tested copy md5 7927a5b8ea646f1cc830975f39bfc015), then build all H-2 deliverables
import hashlib; print('generator md5', hashlib.md5(open('h2_make_report.py','rb').read()).hexdigest())
!python h2_make_report.py --out-dir ./h2_outputs --h1-dir ./h1_outputs 2>&1 | tee -a h2_outputs/scan_log.txt

In [ ]:
print(open('h2_outputs/H2_CSE-CIC-IDS2018_Verification_Report.md').read())

In [ ]:
# Step 9: bundle all H-2 deliverables (scripts included) for download: Files panel -> h2_outputs.zip
!cp h2_ids2018_scan.py h2_make_report.py h2_outputs/ && rm -f h2_outputs.zip && zip -qr h2_outputs.zip h2_outputs && cd h2_outputs && md5sum * && ls -la ../h2_outputs.zip

# H-3 → H-5: labeling rule, feature module, first feature file

* **H-3**: `sentinelai/labeling.py` and `docs/H3_LABELING_RULE.md`. The rule maps exact, protocol-aware labels from `label_info.txt` to normal (0) or suspicious (1). Unmapped labels are never counted as benign. Malak checks the rule.
* **H-4**: `sentinelai/features.py`. It builds per-flow features plus past-only 5-minute windows for each client host. Training and runtime use the same streaming code, and `tests/` proves that no window looks into the future.
* **H-5**: `h5_publish_features.py`. It writes `h5_outputs/features_fox.parquet` and checks it against `feature_contract_v0.json`, which stays a proposal until ALL-1 is signed off.

The code was tested outside Colab first (34 tests pass on pandas 2.2.3 and 3.0). Each `%%writefile` cell is followed by an MD5 check against that tested copy.

In [ ]:
# H-3..H-5 Step 1: project folder (same layout as the repo: sentinelai/ package, tests/, docs/)
import os
for d in ['sentinelai_h3h5/sentinelai', 'sentinelai_h3h5/tests', 'sentinelai_h3h5/docs']:
    os.makedirs(d, exist_ok=True)
open('sentinelai_h3h5/sentinelai/__init__.py', 'w').close()
print(sorted(os.listdir('sentinelai_h3h5')))

In [ ]:
%%writefile sentinelai_h3h5/sentinelai/labeling.py
"""
SentinelAI H-3 - labeling rule: AIT Netflow labels -> normal (0) / suspicious (1).

THE RULE (version 1.0)
  1. The only source of truth is the publisher's label_info.txt
     (Zenodo record 13168643, md5 e32c0e08d15725999df0c6a0c02aff47).
  2. A flow gets y = 0 if its label string is listed under
     "Benign labels in <its protocol>", and y = 1 if it is listed under
     "Malicious labels in <its protocol>".
  3. Matching is EXACT: no whitespace stripping, no case folding, and the
     protocol must match (a UDP-only label found in a TCP file is NOT mapped).
  4. Anything else (empty label, unknown string, label listed only for the other
     protocol) is UNMAPPED: y = <NA>. Unmapped flows are never treated as
     benign. They are left out of training/evaluation labels, but they are
     still real traffic, so the feature module keeps them when it computes the
     per-host time windows.
  5. Protocol comes from the file name: tcp_complete.csv / tcp_nocomplete.csv -> TCP,
     udp_complete.csv -> UDP (H-1 checked this against each file's schema).
"""
from __future__ import annotations

import hashlib
import re
from pathlib import Path

import pandas as pd

RULE_VERSION = "1.0"
OFFICIAL_LABEL_INFO_MD5 = "e32c0e08d15725999df0c6a0c02aff47"
NORMAL, SUSPICIOUS = 0, 1
_SECTION = re.compile(r"(Benign|Malicious) labels in (TCP|UDP):")


def parse_label_info_text(text: str) -> dict[str, dict[str, int]]:
    """Parse label_info.txt text into {"TCP": {label: 0|1}, "UDP": {...}}.
    Raises if the four sections are not all present, if a label appears before
    the first section, or if a label is listed as both benign and malicious."""
    mapping: dict[str, dict[str, int]] = {"TCP": {}, "UDP": {}}
    sections, current = [], None
    for lineno, line in enumerate(text.splitlines(), 1):   # splitlines drops \r\n
        if line.strip() == "":
            continue
        m = _SECTION.fullmatch(line.strip())
        if m:
            current = (m.group(2), NORMAL if m.group(1) == "Benign" else SUSPICIOUS)
            sections.append(m.group(0))
            continue
        if current is None:
            raise ValueError(f"label_info.txt line {lineno}: label before any section: {line!r}")
        proto, y = current
        if mapping[proto].get(line, y) != y:
            raise ValueError(f"label_info.txt line {lineno}: {line!r} is both benign and malicious in {proto}")
        mapping[proto][line] = y
    if sorted(sections) != sorted({f"{k} labels in {p}:" for k in ("Benign", "Malicious") for p in ("TCP", "UDP")}):
        raise ValueError(f"expected the 4 sections, found {sections}")
    return mapping


def load_label_info(path: str | Path, check_md5: bool = True) -> dict[str, dict[str, int]]:
    raw = Path(path).read_bytes()
    if check_md5:
        actual = hashlib.md5(raw).hexdigest()
        if actual != OFFICIAL_LABEL_INFO_MD5:
            raise ValueError(f"label_info.txt md5 {actual} != official {OFFICIAL_LABEL_INFO_MD5}")
    return parse_label_info_text(raw.decode("utf-8-sig"))


def protocol_from_filename(name: str) -> str:
    base = Path(name).name.lower()
    if base.startswith("tcp"):
        return "TCP"
    if base.startswith("udp"):
        return "UDP"
    raise ValueError(f"cannot infer protocol from file name {name!r}")


def apply_rule(labels: pd.Series, protocol: str, mapping: dict[str, dict[str, int]]) -> pd.DataFrame:
    """Return a DataFrame with y (Int8, <NA> = unmapped) and label_status
    ('normal' / 'suspicious' / 'unmapped')."""
    if protocol not in mapping:
        raise ValueError(f"unknown protocol {protocol!r}")
    labels = labels.astype("string").fillna("")          # a real NaN label is unmapped, never benign
    y = labels.map(mapping[protocol]).astype("Int8")      # exact dict lookup
    status = pd.Series("unmapped", index=labels.index, dtype="string")
    status[y == NORMAL] = "normal"
    status[y == SUSPICIOUS] = "suspicious"
    return pd.DataFrame({"y": y, "label_status": status})


In [ ]:
%%writefile sentinelai_h3h5/sentinelai/features.py
"""
SentinelAI H-4 - feature module (one code path for training AND runtime).

Each AIT Netflow row (Tstat tcp_complete / tcp_nocomplete / udp_complete) becomes
one feature row:

* Per-flow features: protocol, handshake completeness, server port, duration,
  packets, bytes and TCP flag counts. Client IP, server IP, client port,
  timestamps, host names (fqdn) and the AIT role/network columns are NOT features.
* Past-only 5-minute window features for the flow's CLIENT host: number of
  connections, distinct server ports, distinct server hosts, DNS flow count and
  DNS bytes. Only flows from the same client host in the same network that ENDED
  in [t - 300 s, t) count, where t is the current flow's end time. A Tstat record
  only exists once a flow has ended, so this is all the Detection Agent could
  have seen when it scores the flow. Flows ending at exactly t (ties) and
  anything later are never used. IPs are kept in the output as info columns
  (src_ip, dst_ip) for evidence and correlation, but they are never model inputs.

Runtime:  ext = FlowFeatureExtractor(); feats = ext.process(flow_dict)   (flows in end-time order)
Training: build_features(flows_df)  -> sorts by end time and calls the SAME process() row by row.
"""
from __future__ import annotations

import io
import re
import zipfile
from collections import Counter, deque
from pathlib import Path

import numpy as np
import pandas as pd

from .labeling import apply_rule, protocol_from_filename

FEATURE_MODULE_VERSION = "1.1"
WINDOW_MS = 300_000            # 5 minutes
DNS_PORT = 53

# ---------------------------------------------------------------------------
# Output columns (the feature contract; see contracts/feature_contract_v0.json)
# ---------------------------------------------------------------------------
META_COLUMNS = ["flow_id", "network", "source_file", "event_end_ms", "src_ip", "dst_ip"]   # info only, never model inputs
LABEL_COLUMNS = ["y", "label_original"]
FLOW_FEATURES = [
    "is_tcp", "tcp_handshake_complete", "dst_port", "dst_port_wellknown",
    "duration_ms", "c_pkts", "s_pkts", "c_bytes", "s_bytes",
    "c_bytes_per_pkt", "s_bytes_per_pkt", "c_bytes_share",
    "c_syn", "c_fin", "c_rst", "s_syn", "s_fin", "s_rst",
    "c_pkts_data", "s_pkts_data", "c_pkts_retx", "s_pkts_retx",
]
WINDOW_FEATURES = [
    "w5_conn_count", "w5_distinct_dst_ports", "w5_distinct_dst_hosts",
    "w5_dns_count", "w5_dns_bytes", "w5_dns_mean_bytes",
]
FEATURE_COLUMNS = FLOW_FEATURES + WINDOW_FEATURES
FLOAT_FEATURES = {"duration_ms", "c_bytes_per_pkt", "s_bytes_per_pkt", "c_bytes_share", "w5_dns_mean_bytes"}

# Raw Tstat columns read from each file type (clean names, see clean_column()).
_TCP_COLS = ["c_ip", "c_port", "s_ip", "s_port", "c_pkts_all", "s_pkts_all", "c_bytes_all", "s_bytes_all",
             "c_syn_cnt", "c_fin_cnt", "c_rst_cnt", "s_syn_cnt", "s_fin_cnt", "s_rst_cnt",
             "c_pkts_data", "s_pkts_data", "c_pkts_retx", "s_pkts_retx", "first", "last", "durat", "label"]
_UDP_COLS = ["c_ip", "c_port", "s_ip", "s_port", "c_first_abs", "c_durat", "c_bytes_all", "c_pkts_all",
             "s_first_abs", "s_durat", "s_bytes_all", "s_pkts_all", "label"]
_TEXT_COLS = {"c_ip", "s_ip", "label"}


def clean_column(raw: str) -> str:
    """'#15#c_ip:1' -> 'c_ip', 'c_port:2' -> 'c_port', 'label' -> 'label' (same rule as H-1)."""
    return re.sub(r":\d+$", "", re.sub(r"^#(\d+#)?", "", raw))


# ---------------------------------------------------------------------------
# 1. Raw Tstat rows -> one normalised flow table
# ---------------------------------------------------------------------------
def normalise_flows(raw: pd.DataFrame, source_file: str, network: str) -> pd.DataFrame:
    """Turn one raw Tstat file (clean column names) into the common flow schema.
    Times are in ms since epoch. TCP 'durat' is ms; UDP 'c_durat'/'s_durat' are
    SECONDS (Tstat documentation), so they are converted here."""
    proto = protocol_from_filename(source_file)
    complete = proto == "TCP" and "nocomplete" not in source_file.lower()
    need = _TCP_COLS if proto == "TCP" else _UDP_COLS
    missing = [c for c in need if c not in raw.columns]
    if missing:
        raise ValueError(f"{network}/{source_file}: missing columns {missing}")

    num = {}
    for c in need:
        if c in _TEXT_COLS:
            continue
        v = pd.to_numeric(raw[c], errors="coerce")
        bad = int(v.isna().sum())
        if bad:
            raise ValueError(f"{network}/{source_file}: {bad} non-numeric/empty values in column {c!r}")
        num[c] = v.to_numpy(dtype="float64")

    n = len(raw)
    out = pd.DataFrame({
        "flow_id": [f"{network}/{Path(source_file).stem}/{i}" for i in range(n)],  # i = 0-based data row
        "network": network,
        "source_file": Path(source_file).name,
        "protocol": proto,
        "c_ip": raw["c_ip"].astype(str).to_numpy(),
        "s_ip": raw["s_ip"].astype(str).to_numpy(),
        "label_original": raw["label"].astype(str).to_numpy(),
    })
    if proto == "TCP":
        start, end = num["first"], num["last"]
        duration = num["durat"]
        tcp = {k: num[f"{k}_cnt" if k[2:] in ("syn", "fin", "rst") else k]
               for k in ("c_syn", "c_fin", "c_rst", "s_syn", "s_fin", "s_rst",
                         "c_pkts_data", "s_pkts_data", "c_pkts_retx", "s_pkts_retx")}
    else:
        c_end = num["c_first_abs"] + num["c_durat"] * 1000.0
        s_has = num["s_pkts_all"] > 0
        s_end = np.where(s_has, num["s_first_abs"] + num["s_durat"] * 1000.0, -np.inf)
        start = np.where(s_has, np.minimum(num["c_first_abs"], num["s_first_abs"]), num["c_first_abs"])
        end = np.maximum(c_end, s_end)
        duration = end - start
        tcp = {k: np.zeros(n) for k in ("c_syn", "c_fin", "c_rst", "s_syn", "s_fin", "s_rst",
                                        "c_pkts_data", "s_pkts_data", "c_pkts_retx", "s_pkts_retx")}
    if np.any(end < start):
        raise ValueError(f"{network}/{source_file}: {int(np.sum(end < start))} flows end before they start")

    out["event_end_ms"] = end
    out["is_tcp"] = int(proto == "TCP")
    out["tcp_handshake_complete"] = int(complete)
    out["dst_port"] = num["s_port"]
    out["duration_ms"] = duration
    out["c_pkts"], out["s_pkts"] = num["c_pkts_all"], num["s_pkts_all"]
    out["c_bytes"], out["s_bytes"] = num["c_bytes_all"], num["s_bytes_all"]
    for k, v in tcp.items():
        out[k] = v
    return out


def read_network_zip(zip_path: str | Path, network: str) -> pd.DataFrame:
    """Stream the three CSVs out of one AIT archive (nothing is extracted) and
    return the normalised flow table for the whole network."""
    parts = []
    with zipfile.ZipFile(zip_path) as zf:
        for member in sorted(m for m in zf.namelist() if m.lower().endswith(".csv")):
            with zf.open(member) as f:
                header = io.TextIOWrapper(f, encoding="utf-8").readline().rstrip("\r\n").split(",")
            clean = [clean_column(h) for h in header]
            if len(set(clean)) != len(clean):
                raise ValueError(f"{member}: duplicate column names after cleaning")
            want = set(_TCP_COLS if protocol_from_filename(member) == "TCP" else _UDP_COLS)
            usecols = [h for h, c in zip(header, clean) if c in want]
            with zf.open(member) as f:
                raw = pd.read_csv(f, usecols=usecols, dtype=str, keep_default_na=False,
                                  na_filter=False, encoding="utf-8")
            raw.columns = [clean_column(c) for c in raw.columns]
            parts.append(normalise_flows(raw, Path(member).name, network))
    return pd.concat(parts, ignore_index=True)


# ---------------------------------------------------------------------------
# 2. Streaming window state (THE code used at runtime and in training)
# ---------------------------------------------------------------------------
class _HostWindow:
    """Flows of one client host that ended in [t - W, t). Flows that ended at the
    current time t wait in pending and only join once time moves past t."""
    __slots__ = ("events", "ports", "hosts", "dns_count", "dns_bytes", "pending_t", "pending")

    def __init__(self):
        self.events = deque()          # (end_ms, dst_port, dst_host, is_dns, bytes)
        self.ports, self.hosts = Counter(), Counter()
        self.dns_count, self.dns_bytes = 0, 0.0
        self.pending_t, self.pending = None, []

    def _admit(self, ev):
        self.events.append(ev)
        self.ports[ev[1]] += 1
        self.hosts[ev[2]] += 1
        if ev[3]:
            self.dns_count += 1
            self.dns_bytes += ev[4]

    def _evict(self, ev):
        for counter, key in ((self.ports, ev[1]), (self.hosts, ev[2])):
            counter[key] -= 1
            if counter[key] == 0:
                del counter[key]
        if ev[3]:
            self.dns_count -= 1
            self.dns_bytes -= ev[4]

    def features_then_add(self, t, ev, window_ms):
        if self.pending_t is not None and self.pending_t < t:   # time moved on: ties become "past"
            for p in self.pending:
                self._admit(p)
            self.pending_t, self.pending = None, []
        while self.events and self.events[0][0] < t - window_ms:
            self._evict(self.events.popleft())
        n = len(self.events)
        feats = (n, len(self.ports), len(self.hosts), self.dns_count,
                 self.dns_bytes if self.dns_count else 0.0,   # avoids float residue after eviction
                 self.dns_bytes / self.dns_count if self.dns_count else 0.0)
        self.pending_t = t
        self.pending.append(ev)
        return feats


class FlowFeatureExtractor:
    """Feed flows in non-decreasing event_end_ms order (per network); get features back.
    A flow dict needs: network, c_ip, s_ip, event_end_ms, protocol fields from
    normalise_flows() (is_tcp, tcp_handshake_complete, dst_port, duration_ms,
    c_pkts, s_pkts, c_bytes, s_bytes and the TCP counters)."""

    def __init__(self, window_ms: int = WINDOW_MS):
        self.window_ms = window_ms
        self._hosts: dict[tuple, _HostWindow] = {}
        self._last_t: dict[str, float] = {}

    def process(self, flow: dict) -> dict:
        net, t = flow["network"], float(flow["event_end_ms"])
        if t < self._last_t.get(net, -np.inf):
            raise ValueError(f"flow {flow.get('flow_id')} arrived out of time order "
                             f"({t} < {self._last_t[net]}); sort or buffer flows by end time first")
        self._last_t[net] = t
        key = (net, flow["c_ip"])
        hw = self._hosts.get(key)
        if hw is None:
            hw = self._hosts[key] = _HostWindow()
        port = int(flow["dst_port"])
        is_dns = port == DNS_PORT
        nbytes = float(flow["c_bytes"]) + float(flow["s_bytes"])
        w = hw.features_then_add(t, (t, port, flow["s_ip"], is_dns, nbytes), self.window_ms)
        return {**flow_features(flow), **dict(zip(WINDOW_FEATURES, w))}


def flow_features(flow: dict) -> dict:
    """Per-flow features only (no history)."""
    cp, sp = float(flow["c_pkts"]), float(flow["s_pkts"])
    cb, sb = float(flow["c_bytes"]), float(flow["s_bytes"])
    out = {
        "is_tcp": int(flow["is_tcp"]),
        "tcp_handshake_complete": int(flow["tcp_handshake_complete"]),
        "dst_port": int(flow["dst_port"]),
        "dst_port_wellknown": int(int(flow["dst_port"]) < 1024),
        "duration_ms": float(flow["duration_ms"]),
        "c_pkts": int(cp), "s_pkts": int(sp), "c_bytes": int(cb), "s_bytes": int(sb),
        "c_bytes_per_pkt": cb / cp if cp > 0 else 0.0,
        "s_bytes_per_pkt": sb / sp if sp > 0 else 0.0,
        "c_bytes_share": cb / (cb + sb) if (cb + sb) > 0 else 0.0,
    }
    for k in ("c_syn", "c_fin", "c_rst", "s_syn", "s_fin", "s_rst",
              "c_pkts_data", "s_pkts_data", "c_pkts_retx", "s_pkts_retx"):
        out[k] = int(flow[k])
    return out


# ---------------------------------------------------------------------------
# 3. Batch (training) entry point - wraps the same streaming extractor
# ---------------------------------------------------------------------------
_FLOW_INPUT = ["network", "flow_id", "c_ip", "s_ip", "event_end_ms", "is_tcp", "tcp_handshake_complete",
               "dst_port", "duration_ms", "c_pkts", "s_pkts", "c_bytes", "s_bytes",
               "c_syn", "c_fin", "c_rst", "s_syn", "s_fin", "s_rst",
               "c_pkts_data", "s_pkts_data", "c_pkts_retx", "s_pkts_retx"]


def build_features(flows: pd.DataFrame, window_ms: int = WINDOW_MS) -> pd.DataFrame:
    """flows: output of normalise_flows()/read_network_zip() (any row order, any
    number of networks). Returns META + FEATURE columns, one row per input flow,
    in the input's row order. Rows are processed in (network, event_end_ms, flow_id)
    order; the result does not depend on the input order."""
    order = flows.sort_values(["network", "event_end_ms", "flow_id"], kind="mergesort").index
    ext = FlowFeatureExtractor(window_ms)
    cols = {c: flows[c].to_numpy() for c in _FLOW_INPUT}
    pos = flows.index.get_indexer(order)
    rows = [None] * len(flows)
    for p in pos:
        rows[p] = ext.process({c: cols[c][p] for c in _FLOW_INPUT})
    feats = pd.DataFrame.from_records(rows, columns=FEATURE_COLUMNS, index=flows.index)
    for c in FEATURE_COLUMNS:
        feats[c] = feats[c].astype("float64" if c in FLOAT_FEATURES else "int64")
    meta = flows[["flow_id", "network", "source_file", "event_end_ms", "c_ip", "s_ip"]].rename(
        columns={"c_ip": "src_ip", "s_ip": "dst_ip"})
    meta["event_end_ms"] = meta["event_end_ms"].astype("float64")
    return pd.concat([meta, feats], axis=1)


def build_labeled_features(flows: pd.DataFrame, label_map: dict, window_ms: int = WINDOW_MS):
    """Features for ALL flows (unmapped ones still shape the windows), then the
    H-3 rule; unmapped rows are dropped from the returned table and counted."""
    feats = build_features(flows, window_ms)
    y = pd.Series(pd.NA, index=flows.index, dtype="Int8")
    for proto in ("TCP", "UDP"):
        m = flows["protocol"] == proto
        if m.any():
            y[m] = apply_rule(flows.loc[m, "label_original"], proto, label_map)["y"]
    feats["y"] = y
    feats["label_original"] = flows["label_original"].astype(str)
    unmapped = feats[feats["y"].isna()]
    labeled = feats[feats["y"].notna()].copy()
    labeled["y"] = labeled["y"].astype("int8")
    return labeled[META_COLUMNS + LABEL_COLUMNS + FEATURE_COLUMNS].reset_index(drop=True), unmapped


In [ ]:
%%writefile sentinelai_h3h5/sentinelai/contract.py
"""
SentinelAI - feature contract (ALL-1 part a) and a validator for feature files.

The contract is generated from features.py so the two can never drift apart;
write_contract() writes it as JSON for the repo, validate() checks a table.
"""
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

from . import features as F
from .labeling import RULE_VERSION

CONTRACT_VERSION = "0.2"
FORBIDDEN_FEATURE_PATTERNS = [r"ip", r"c_port", r"time", r"first", r"last", r"fqdn", r"role", r"network",
                              r"label", r"^y$", r"flow_id", r"source_file", r"event_end"]

DESCRIPTIONS = {
    "flow_id": "Unique event ID '<network>/<file stem>/<0-based data row>'; use in Alert evidence refs",
    "network": "AIT network name; for splitting (H-6) only",
    "source_file": "tcp_complete.csv / tcp_nocomplete.csv / udp_complete.csv",
    "event_end_ms": "Flow end time, ms since Unix epoch (UTC); for ordering/evidence only",
    "src_ip": "Client (source) IP address; info for evidence/correlation, never a model input",
    "dst_ip": "Server (destination) IP address; info for evidence/correlation, never a model input",
    "y": "0 = normal, 1 = suspicious (H-3 rule; unmapped rows are not in the file)",
    "label_original": "Exact AIT label string, for audit only",
    "is_tcp": "1 TCP, 0 UDP",
    "tcp_handshake_complete": "1 if from tcp_complete.csv, else 0 (UDP is always 0)",
    "dst_port": "Server port", "dst_port_wellknown": "1 if dst_port < 1024",
    "duration_ms": "Flow duration in ms (UDP converted from seconds)",
    "c_pkts": "Client->server packets", "s_pkts": "Server->client packets",
    "c_bytes": "Client->server bytes", "s_bytes": "Server->client bytes",
    "c_bytes_per_pkt": "c_bytes / c_pkts (0 if no packets)", "s_bytes_per_pkt": "s_bytes / s_pkts (0 if no packets)",
    "c_bytes_share": "c_bytes / (c_bytes + s_bytes) (0 if no bytes)",
    "c_syn": "Client SYN count (TCP; 0 for UDP)", "c_fin": "Client FIN count", "c_rst": "Client RST count",
    "s_syn": "Server SYN count", "s_fin": "Server FIN count", "s_rst": "Server RST count",
    "c_pkts_data": "Client data packets (TCP)", "s_pkts_data": "Server data packets (TCP)",
    "c_pkts_retx": "Client retransmitted packets (TCP)", "s_pkts_retx": "Server retransmitted packets (TCP)",
    "w5_conn_count": "Flows from the same client host that ended in [t-300 s, t)",
    "w5_distinct_dst_ports": "Distinct server ports among those flows",
    "w5_distinct_dst_hosts": "Distinct server hosts among those flows",
    "w5_dns_count": "Those flows with dst_port 53",
    "w5_dns_bytes": "Total bytes (both directions) of those DNS flows",
    "w5_dns_mean_bytes": "w5_dns_bytes / w5_dns_count (0 if none)",
}


def dtype_of(col: str) -> str:
    if col in ("flow_id", "network", "source_file", "label_original", "src_ip", "dst_ip"):
        return "string"
    if col == "y":
        return "int8"
    if col == "event_end_ms":
        return "float64"
    return "float64" if col in F.FLOAT_FEATURES else "int64"


def contract_dict() -> dict:
    def cols(names, role):
        return [{"name": c, "dtype": dtype_of(c), "role": role, "description": DESCRIPTIONS[c]} for c in names]
    return {
        "contract": "SentinelAI feature table",
        "contract_version": CONTRACT_VERSION,
        "status": "PROPOSED by Habiba (H-4/H-5); becomes final when ALL-1 is signed off",
        "feature_module_version": F.FEATURE_MODULE_VERSION,
        "labeling_rule_version": RULE_VERSION,
        "window": {"length_ms": F.WINDOW_MS, "host_key": "client IP within one network (key only, never output)",
                   "interval": "[t - 300 s, t), t = current flow end time; ties at t excluded"},
        "column_order": F.META_COLUMNS + F.LABEL_COLUMNS + F.FEATURE_COLUMNS,
        "model_input_columns": F.FEATURE_COLUMNS,
        "columns": cols(F.META_COLUMNS, "meta (never a model input)")
                   + cols(F.LABEL_COLUMNS, "label (never a model input)")
                   + cols(F.FEATURE_COLUMNS, "feature"),
    }


def write_contract(path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(contract_dict(), indent=2) + "\n", encoding="utf-8")
    return path


def validate(df: pd.DataFrame, contract: dict | None = None) -> list[str]:
    """Return a list of problems ([] means the table matches the contract)."""
    c = contract or contract_dict()
    problems = []
    if list(df.columns) != c["column_order"]:
        problems.append(f"column order differs from contract: {list(df.columns)}")
        return problems
    for spec in c["columns"]:
        s, want = df[spec["name"]], spec["dtype"]
        ok = (pd.api.types.is_string_dtype(s) or s.dtype == object) if want == "string" else str(s.dtype) == want
        if not ok:
            problems.append(f"{spec['name']}: dtype {s.dtype}, contract says {want}")
    for name in c["model_input_columns"]:
        if any(re.search(p, name) for p in FORBIDDEN_FEATURE_PATTERNS):
            problems.append(f"model input {name!r} looks like an ID/IP/time/label column")
    X = df[c["model_input_columns"]].to_numpy(dtype="float64")
    if not np.isfinite(X).all():
        problems.append(f"{int((~np.isfinite(X)).sum())} NaN/inf values in model input columns")
    if (X < 0).any():
        bad = [n for n in c["model_input_columns"] if (df[n] < 0).any()]
        problems.append(f"negative values in {bad}")
    if not set(df["y"].unique()) <= {0, 1}:
        problems.append(f"y has values {sorted(df['y'].unique())}")
    if df["flow_id"].duplicated().any():
        problems.append(f"{int(df['flow_id'].duplicated().sum())} duplicate flow_id values")
    return problems


In [ ]:
%%writefile sentinelai_h3h5/tests/real_headers.py
"""Exact raw header lines of the three AIT Netflow file types, copied from the
H-1 Step 4 output (identical in all 8 networks), plus label_info.txt verbatim."""
TCP_COMPLETE = "#15#c_ip:1,c_port:2,c_pkts_all:3,c_rst_cnt:4,c_ack_cnt:5,c_ack_cnt_p:6,c_bytes_uniq:7,c_pkts_data:8,c_bytes_all:9,c_pkts_retx:10,c_bytes_retx:11,c_pkts_ooo:12,c_syn_cnt:13,c_fin_cnt:14,s_ip:15,s_port:16,s_pkts_all:17,s_rst_cnt:18,s_ack_cnt:19,s_ack_cnt_p:20,s_bytes_uniq:21,s_pkts_data:22,s_bytes_all:23,s_pkts_retx:24,s_bytes_retx:25,s_pkts_ooo:26,s_syn_cnt:27,s_fin_cnt:28,first:29,last:30,durat:31,c_first:32,s_first:33,c_last:34,s_last:35,c_first_ack:36,s_first_ack:37,c_isint:38,s_isint:39,c_iscrypto:40,s_iscrypto:41,con_t:42,p2p_t:43,http_t:44,c_rtt_avg:45,c_rtt_min:46,c_rtt_max:47,c_rtt_std:48,c_rtt_cnt:49,c_ttl_min:50,c_ttl_max:51,s_rtt_avg:52,s_rtt_min:53,s_rtt_max:54,s_rtt_std:55,s_rtt_cnt:56,s_ttl_min:57,s_ttl_max:58,p2p_st:59,ed2k_data:60,ed2k_sig:61,ed2k_c2s:62,ed2k_c2c:63,ed2k_chat:64,c_f1323_opt:65,c_tm_opt:66,c_win_scl:67,c_sack_opt:68,c_sack_cnt:69,c_mss:70,c_mss_max:71,c_mss_min:72,c_win_max:73,c_win_min:74,c_win_0:75,c_cwin_max:76,c_cwin_min:77,c_cwin_ini:78,c_pkts_rto:79,c_pkts_fs:80,c_pkts_reor:81,c_pkts_dup:82,c_pkts_unk:83,c_pkts_fc:84,c_pkts_unrto:85,c_pkts_unfs:86,c_syn_retx:87,s_f1323_opt:88,s_tm_opt:89,s_win_scl:90,s_sack_opt:91,s_sack_cnt:92,s_mss:93,s_mss_max:94,s_mss_min:95,s_win_max:96,s_win_min:97,s_win_0:98,s_cwin_max:99,s_cwin_min:100,s_cwin_ini:101,s_pkts_rto:102,s_pkts_fs:103,s_pkts_reor:104,s_pkts_dup:105,s_pkts_unk:106,s_pkts_fc:107,s_pkts_unrto:108,s_pkts_unfs:109,s_syn_retx:110,http_req_cnt:111,http_res_cnt:112,http_res:113,c_pkts_push:114,s_pkts_push:115,c_tls_SNI:116,s_tls_SCN:117,c_npnalpn:118,s_npnalpn:119,c_tls_sesid:120,c_last_handshakeT:121,s_last_handshakeT:122,c_appdataT:123,s_appdataT:124,c_appdataB:125,s_appdataB:126,fqdn:127,dns_rslv:128,req_tm:129,res_tm:130,timestamp,role_cli,ipv4_address_cli,network_cli,role_serv,ipv4_address_serv,network_serv,label"
TCP_NOCOMPLETE = "#15#c_ip:1,c_port:2,c_pkts_all:3,c_rst_cnt:4,c_ack_cnt:5,c_ack_cnt_p:6,c_bytes_uniq:7,c_pkts_data:8,c_bytes_all:9,c_pkts_retx:10,c_bytes_retx:11,c_pkts_ooo:12,c_syn_cnt:13,c_fin_cnt:14,s_ip:15,s_port:16,s_pkts_all:17,s_rst_cnt:18,s_ack_cnt:19,s_ack_cnt_p:20,s_bytes_uniq:21,s_pkts_data:22,s_bytes_all:23,s_pkts_retx:24,s_bytes_retx:25,s_pkts_ooo:26,s_syn_cnt:27,s_fin_cnt:28,first:29,last:30,durat:31,c_first:32,s_first:33,c_last:34,s_last:35,c_first_ack:36,s_first_ack:37,c_isint:38,s_isint:39,c_iscrypto:40,s_iscrypto:41,con_t:42,p2p_t:43,http_t:44,timestamp,role_cli,ipv4_address_cli,network_cli,role_serv,ipv4_address_serv,network_serv,label"
UDP_COMPLETE = "#c_ip:1,c_port:2,c_first_abs:3,c_durat:4,c_bytes_all:5,c_pkts_all:6,c_isint:7,c_iscrypto:8,c_type:9,s_ip:10,s_port:11,s_first_abs:12,s_durat:13,s_bytes_all:14,s_pkts_all:15,s_isint:16,s_iscrypto:17,s_type:18,fqdn:19,timestamp,rounded_timestamp,role_cli,ipv4_address_cli,network_cli,role_serv,ipv4_address_serv,network_serv,label"
LABEL_INFO = "\r\n".join([
 "Benign labels in TCP:", "browsing/update", "benign_share", "broken flow - benign", "mail", "monitoring",
 "HTTP(S) intra", "HTTP(S) DMZ", "HTTP", "HTTPS", "DNS", "SSH", "proxy", "update/command on unassigned port", "",
 "Malicious labels in TCP:", "service_scan", "online_cracking", "host_discover_dmz", "host_discover_local",
 "wpscan", "dirb_scan", "upload_rce_shell", "check_user_id", "check_netstat_t", "read_resolv",
 "check_network_config", "check_ps_a", "check_release", "read_group", "read_passwd", "check_date",
 "list_web_dir", "check_wp_config", "dump_wp_users", "read_profile", "dns_brute_force_start", "list_www",
 "check_who", "clear", "check_last", "check_id", "vpn_connect", "check_whoami", "check_uname_r",
 "check_meminfo", "check_uname_a", "check_df", "check_netstat_nat", "list_home", "check_netstat_l",
 "check_cpuinfo", "check_uptime", "check_pwd", "list_l", "",
 "Benign labels in UDP:", "mDNSv6", "mDNS", "bootp", "NTP", "systemd-resolver (local)",
 "benign DNS - attacker requesting updates", "benign DNS", "OpenVPN", "QUIC", "STUN VoIP", "NetBIOS", "GoogleDNS", "",
 "Malicious labels in UDP:", "data exfiltration"])


In [ ]:
%%writefile sentinelai_h3h5/tests/conftest.py
import io
import sys
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import pytest

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "tests"))

import real_headers as RH  # noqa: E402
from sentinelai.features import clean_column  # noqa: E402

T0 = 1_642_204_800_000.0   # 2022-01-15 00:00:00 UTC in ms


def _raw_rows(header: str, rows: list[dict]) -> str:
    """CSV text with the REAL header; unspecified fields are filled with '0'."""
    raw = header.split(",")
    clean = [clean_column(h) for h in raw]
    lines = [header]
    for r in rows:
        lines.append(",".join(str(r.get(c, "0")) for c in clean))
    return "\r\n".join(lines) + "\r\n"


def tcp_row(c_ip, s_ip, s_port, first, durat_ms, label, c_bytes=100, s_bytes=200, c_pkts=3, s_pkts=4, syn=1):
    return {"c_ip": c_ip, "c_port": 40000, "s_ip": s_ip, "s_port": s_port, "first": first,
            "last": first + durat_ms, "durat": durat_ms, "c_bytes_all": c_bytes, "s_bytes_all": s_bytes,
            "c_pkts_all": c_pkts, "s_pkts_all": s_pkts, "c_syn_cnt": syn, "s_syn_cnt": syn,
            "role_cli": "attacker_0", "label": label}


def udp_row(c_ip, s_ip, s_port, first, durat_s, label, c_bytes=60, s_bytes=90, c_pkts=1, s_pkts=1):
    return {"c_ip": c_ip, "c_port": 5353, "s_ip": s_ip, "s_port": s_port, "c_first_abs": first,
            "c_durat": durat_s, "c_bytes_all": c_bytes, "c_pkts_all": c_pkts,
            "s_first_abs": first + 1 if s_pkts else 0, "s_durat": durat_s if s_pkts else 0,
            "s_bytes_all": s_bytes, "s_pkts_all": s_pkts, "label": label}


def make_zip(path: Path, tcp: list[dict], nocomplete: list[dict], udp: list[dict]) -> Path:
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("tcp_complete.csv", _raw_rows(RH.TCP_COMPLETE, tcp))
        z.writestr("tcp_nocomplete.csv", _raw_rows(RH.TCP_NOCOMPLETE, nocomplete))
        z.writestr("udp_complete.csv", _raw_rows(RH.UDP_COMPLETE, udp))
    return path


def random_flows(n=1500, hosts=12, seed=0, network="net", span_ms=40 * 60_000, tie_every=7):
    """Normalised flow table (as normalise_flows() returns) with many same-time ties."""
    rng = np.random.default_rng(seed)
    t = np.sort(T0 + rng.integers(0, span_ms, n).astype(float))
    t[::tie_every] = t[np.maximum(np.arange(0, n, tie_every) - 1, 0)]     # force exact ties
    is_tcp = rng.random(n) < 0.6
    port = np.where(rng.random(n) < 0.25, 53, rng.choice([22, 25, 80, 443, 8080, 3306, 6000], n))
    df = pd.DataFrame({
        "flow_id": [f"{network}/x/{i}" for i in range(n)], "network": network,
        "source_file": np.where(is_tcp, "tcp_complete.csv", "udp_complete.csv"),
        "protocol": np.where(is_tcp, "TCP", "UDP"),
        "c_ip": rng.choice([f"10.0.0.{i}" for i in range(hosts)], n),
        "s_ip": rng.choice([f"192.168.1.{i}" for i in range(30)], n),
        "label_original": "x", "event_end_ms": t,
        "is_tcp": is_tcp.astype(int), "tcp_handshake_complete": is_tcp.astype(int),
        "dst_port": port.astype(float), "duration_ms": rng.integers(0, 5000, n).astype(float),
        "c_pkts": rng.integers(0, 50, n).astype(float), "s_pkts": rng.integers(0, 50, n).astype(float),
        "c_bytes": rng.integers(0, 9000, n).astype(float), "s_bytes": rng.integers(0, 9000, n).astype(float),
    })
    for k in ("c_syn", "c_fin", "c_rst", "s_syn", "s_fin", "s_rst",
              "c_pkts_data", "s_pkts_data", "c_pkts_retx", "s_pkts_retx"):
        df[k] = np.where(is_tcp, rng.integers(0, 4, n), 0).astype(float)
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)   # shuffled on purpose


@pytest.fixture
def label_map():
    from sentinelai.labeling import parse_label_info_text
    return parse_label_info_text(RH.LABEL_INFO)


In [ ]:
%%writefile sentinelai_h3h5/tests/test_labeling.py
import hashlib

import pandas as pd
import pytest

import real_headers as RH
from sentinelai.labeling import (OFFICIAL_LABEL_INFO_MD5, apply_rule, load_label_info,
                                 parse_label_info_text, protocol_from_filename)


def test_fixture_is_the_official_file():
    assert hashlib.md5(RH.LABEL_INFO.encode()).hexdigest() == OFFICIAL_LABEL_INFO_MD5


def test_section_sizes(label_map):
    count = lambda p, y: sum(v == y for v in label_map[p].values())
    assert (count("TCP", 0), count("TCP", 1), count("UDP", 0), count("UDP", 1)) == (13, 39, 12, 1)


def test_known_labels(label_map):
    s = pd.Series(["service_scan", "mail", "broken flow - benign", "dns_brute_force_start"])
    assert apply_rule(s, "TCP", label_map)["y"].tolist() == [1, 0, 0, 1]
    s = pd.Series(["data exfiltration", "benign DNS - attacker requesting updates", "NTP"])
    assert apply_rule(s, "UDP", label_map)["y"].tolist() == [1, 0, 0]


@pytest.mark.parametrize("value", ["", " mail", "mail ", "MAIL", "Service_scan", "unknown", "nan"])
def test_unmapped_is_never_benign(label_map, value):
    out = apply_rule(pd.Series([value]), "TCP", label_map)
    assert out["y"].isna().all() and (out["label_status"] == "unmapped").all()


def test_real_nan_is_unmapped(label_map):
    out = apply_rule(pd.Series([None, float("nan")], dtype=object), "TCP", label_map)
    assert out["y"].isna().all()


def test_protocol_must_match(label_map):
    assert apply_rule(pd.Series(["data exfiltration"]), "TCP", label_map)["y"].isna().all()
    assert apply_rule(pd.Series(["DNS"]), "UDP", label_map)["y"].isna().all()   # 'DNS' is TCP-only


def test_protocol_from_filename():
    assert protocol_from_filename("tcp_complete.csv") == "TCP"
    assert protocol_from_filename("x/tcp_nocomplete.csv") == "TCP"
    assert protocol_from_filename("udp_complete.csv") == "UDP"
    with pytest.raises(ValueError):
        protocol_from_filename("flows.csv")


def test_bad_files_are_rejected(tmp_path):
    with pytest.raises(ValueError):
        parse_label_info_text("Benign labels in TCP:\nmail\n")                  # sections missing
    with pytest.raises(ValueError):
        parse_label_info_text("mail\n" + RH.LABEL_INFO)                         # label before section
    p = tmp_path / "label_info.txt"
    p.write_bytes(RH.LABEL_INFO.encode() + b"x")
    with pytest.raises(ValueError):
        load_label_info(p)                                                       # md5 mismatch


In [ ]:
%%writefile sentinelai_h3h5/tests/test_features.py
"""H-4 'done when': tests prove the windows never look into the future."""
import numpy as np
import pandas as pd
import pytest

from conftest import T0, make_zip, random_flows, tcp_row, udp_row
from sentinelai import features as F
from sentinelai.contract import contract_dict, validate
from sentinelai.labeling import parse_label_info_text
import real_headers as RH

W = F.WINDOW_MS
TCP_COUNTERS = ("c_syn", "c_fin", "c_rst", "s_syn", "s_fin", "s_rst",
                "c_pkts_data", "s_pkts_data", "c_pkts_retx", "s_pkts_retx")


def naive_windows(flows: pd.DataFrame) -> pd.DataFrame:
    """Reference implementation straight from the definition, O(n^2):
    same network, same client host, end time in [t - W, t)."""
    out = []
    for _, r in flows.iterrows():
        t = r.event_end_ms
        p = flows[(flows.network == r.network) & (flows.c_ip == r.c_ip)
                  & (flows.event_end_ms >= t - W) & (flows.event_end_ms < t)]
        dns = p[p.dst_port == 53]
        b = float((dns.c_bytes + dns.s_bytes).sum())
        out.append({"w5_conn_count": len(p), "w5_distinct_dst_ports": p.dst_port.nunique(),
                    "w5_distinct_dst_hosts": p.s_ip.nunique(), "w5_dns_count": len(dns),
                    "w5_dns_bytes": b, "w5_dns_mean_bytes": b / len(dns) if len(dns) else 0.0})
    return pd.DataFrame(out, index=flows.index)


def one_flow(i, t, host="h", port=80, network="n"):
    return {"flow_id": f"{network}/x/{i}", "network": network, "source_file": "tcp_complete.csv",
            "protocol": "TCP", "c_ip": host, "s_ip": f"s{i}", "label_original": "x", "event_end_ms": float(t),
            "is_tcp": 1, "tcp_handshake_complete": 1, "dst_port": port, "duration_ms": 1.0, "c_pkts": 1,
            "s_pkts": 1, "c_bytes": 10, "s_bytes": 10, **{k: 0 for k in TCP_COUNTERS}}


# ---------------------------------------------------------------- definition
@pytest.mark.parametrize("seed", [0, 1, 2])
def test_matches_definition(seed):
    flows = random_flows(n=700, seed=seed)
    got = F.build_features(flows)[F.WINDOW_FEATURES].astype(float)
    pd.testing.assert_frame_equal(got, naive_windows(flows).astype(float))


def test_window_boundaries():
    t = T0 + 10 * W
    flows = pd.DataFrame([one_flow(0, t - W - 1), one_flow(1, t - W), one_flow(2, t - 1),
                          one_flow(3, t), one_flow(4, t), one_flow(5, t + 1)])
    w = F.build_features(flows).set_index("flow_id")["w5_conn_count"]
    assert w["n/x/0"] == 0
    assert w["n/x/3"] == 2 and w["n/x/4"] == 2   # t-W (kept) and t-1; t-W-1 too old; the other flow at t is a tie
    assert w["n/x/5"] == 3                       # window [t+1-W, t+1): t-1, t, t


# ---------------------------------------------------------------- no future
@pytest.mark.parametrize("seed", [0, 1, 2, 3])
def test_removing_the_future_changes_nothing(seed):
    flows = random_flows(n=2000, seed=seed)
    full = F.build_features(flows).set_index("flow_id")
    for q in (0.1, 0.35, 0.6, 0.9):
        cut = flows.event_end_ms.quantile(q)
        past = flows[flows.event_end_ms < cut]
        part = F.build_features(past).set_index("flow_id")
        pd.testing.assert_frame_equal(part, full.loc[part.index])


@pytest.mark.parametrize("seed", [0, 1])
def test_changing_the_future_changes_nothing(seed):
    """Rewrite every flow at or after a cut (ports, bytes, hosts) and add a flood
    of future DNS flows from the same hosts; features before the cut must not move."""
    rng = np.random.default_rng(seed)
    flows = random_flows(n=2000, seed=seed)
    cut = flows.event_end_ms.quantile(0.5)
    before = F.build_features(flows).set_index("flow_id")
    fut = flows.event_end_ms >= cut
    changed = flows.copy()
    changed.loc[fut, "dst_port"] = rng.integers(1, 65535, fut.sum()).astype(float)
    changed.loc[fut, "c_bytes"] = 1e6
    changed.loc[fut, "s_ip"] = "evil"
    flood = pd.concat([changed[fut]] * 3, ignore_index=True)
    flood["flow_id"] = [f"net/flood/{i}" for i in range(len(flood))]
    flood["dst_port"] = 53.0
    flood["event_end_ms"] = cut + rng.integers(0, 3 * W, len(flood))
    after = F.build_features(pd.concat([changed, flood], ignore_index=True)).set_index("flow_id")
    past_ids = flows.loc[~fut, "flow_id"]
    pd.testing.assert_frame_equal(after.loc[past_ids], before.loc[past_ids])


def test_ties_do_not_see_each_other():
    """Many flows from one host ending in the same millisecond all get the same
    features, which only count strictly earlier flows."""
    t = T0 + W
    flows = pd.DataFrame([one_flow(0, t - 5)] + [one_flow(i, t, port=1000 + i) for i in range(1, 50)])
    w = F.build_features(flows)
    assert (w.loc[w.flow_id != "n/x/0", "w5_conn_count"] == 1).all()
    assert (w.loc[w.flow_id != "n/x/0", "w5_distinct_dst_ports"] == 1).all()


# ---------------------------------------------------------------- one code path
def test_runtime_equals_batch():
    flows = random_flows(n=1500, seed=5)
    batch = F.build_features(flows).set_index("flow_id")[F.FEATURE_COLUMNS]
    ext = F.FlowFeatureExtractor()
    live = {}
    for _, r in flows.sort_values(["event_end_ms", "flow_id"]).iterrows():   # one flow at a time
        live[r.flow_id] = ext.process(r.to_dict())
    live = pd.DataFrame.from_dict(live, orient="index")[F.FEATURE_COLUMNS].loc[batch.index]
    pd.testing.assert_frame_equal(live.astype(float), batch.astype(float), check_names=False)


def test_runtime_refuses_out_of_order_flows():
    ext = F.FlowFeatureExtractor()
    ext.process(one_flow(0, T0 + 1000))
    with pytest.raises(ValueError, match="out of time order"):
        ext.process(one_flow(1, T0 + 999))


def test_input_row_order_does_not_matter():
    flows = random_flows(n=1200, seed=7)
    a = F.build_features(flows).set_index("flow_id").sort_index()
    b = F.build_features(flows.sample(frac=1, random_state=99)).set_index("flow_id").sort_index()
    pd.testing.assert_frame_equal(a, b)


# ---------------------------------------------------------------- no IPs, no timestamps
def test_no_ip_or_time_columns_in_features():
    banned = ("ip", "time", "first", "last", "fqdn", "role", "c_port", "network", "label", "flow_id")
    assert not [c for c in F.FEATURE_COLUMNS if any(b in c for b in banned)]
    assert validate(F.build_labeled_features(random_flows(300).assign(label_original="mail", protocol="TCP"),
                                             parse_label_info_text(RH.LABEL_INFO))[0]) == []


def test_ips_are_only_grouping_keys():
    """Renaming every IP (one-to-one) and shifting all times by a constant must give identical features."""
    flows = random_flows(n=1200, seed=8)
    base = F.build_features(flows).set_index("flow_id")[F.FEATURE_COLUMNS]
    ren = flows.copy()
    ren["c_ip"] = "X" + ren["c_ip"].str[::-1]
    ren["s_ip"] = "Y" + ren["s_ip"].str[::-1]
    ren["event_end_ms"] += 123_456_789.0
    other = F.build_features(ren).set_index("flow_id")[F.FEATURE_COLUMNS]
    pd.testing.assert_frame_equal(base, other)


def test_networks_do_not_share_windows():
    a = random_flows(n=600, seed=1, network="a")
    b = random_flows(n=600, seed=2, network="b")
    together = F.build_features(pd.concat([a, b], ignore_index=True)).set_index("flow_id")
    alone = pd.concat([F.build_features(a), F.build_features(b)]).set_index("flow_id")
    pd.testing.assert_frame_equal(together.loc[alone.index], alone)


# ---------------------------------------------------------------- raw AIT files
def test_real_headers_end_to_end(tmp_path, label_map):
    t = T0 + 3 * W
    z = make_zip(tmp_path / "toy_netflows.zip",
                 tcp=[tcp_row("10.0.0.1", "10.0.0.9", 80, t - 60_000, 500, "browsing/update"),
                      tcp_row("10.0.0.1", "10.0.0.9", 443, t - 30_000, 2500, "")],    # empty label, ends before the scan
                 nocomplete=[tcp_row("10.0.0.1", "10.0.0.7", 22, t + 1000, 3, "service_scan", s_bytes=0)],
                 udp=[udp_row("10.0.0.1", "10.0.0.53", 53, t - 120_000, 0.25, "benign DNS"),
                      udp_row("10.0.0.2", "10.0.0.53", 53, t, 0.0, "data exfiltration", s_pkts=0)])
    flows = F.read_network_zip(z, "toy")
    assert len(flows) == 5
    labeled, unmapped = F.build_labeled_features(flows, label_map)
    assert validate(labeled) == []
    assert unmapped.flow_id.tolist() == ["toy/tcp_complete/1"]
    r = labeled.set_index("flow_id")
    assert r.loc["toy/udp_complete/0", "duration_ms"] == pytest.approx(251.0)   # 0.25 s -> ms, reply starts 1 ms later
    assert r.loc["toy/udp_complete/1", "duration_ms"] == 0.0                   # no server packets
    assert r.loc["toy/tcp_complete/0", "duration_ms"] == 500.0
    # scan flow: two earlier flows of 10.0.0.1 are complete; the unmapped one still counts as traffic
    s = r.loc["toy/tcp_nocomplete/0"]
    assert (s.w5_conn_count, s.w5_distinct_dst_ports, s.w5_dns_count, s.tcp_handshake_complete) == (3, 3, 1, 0)
    assert s.w5_dns_bytes == 150 and s.y == 1
    assert r.loc["toy/udp_complete/1", "y"] == 1 and r.loc["toy/tcp_complete/0", "y"] == 0
    assert (s.src_ip, s.dst_ip) == ("10.0.0.1", "10.0.0.7")                   # IPs kept as info columns
    assert "src_ip" not in contract_dict()["model_input_columns"]


def test_non_numeric_values_fail_loudly(tmp_path):
    bad = tcp_row("a", "b", 80, T0, 5, "mail")
    bad["c_bytes_all"] = "-"
    with pytest.raises(ValueError, match="non-numeric"):
        F.read_network_zip(make_zip(tmp_path / "bad.zip", [bad], [], []), "bad")


def test_contract_matches_module():
    c = contract_dict()
    assert c["model_input_columns"] == F.FEATURE_COLUMNS
    assert c["column_order"] == F.META_COLUMNS + F.LABEL_COLUMNS + F.FEATURE_COLUMNS


In [ ]:
%%writefile sentinelai_h3h5/h5_publish_features.py
#!/usr/bin/env python3
"""
SentinelAI H-5 - publish a first feature file from ONE network for Malak (M-3).

  python h5_publish_features.py --data-dir ./sentinelai_ait_data --h1-dir ./h1_outputs \
                                --network fox --out-dir ./h5_outputs

Steps: verify the archive + label_info.txt against the official MD5s recorded by
H-1 -> stream the 3 CSVs out of the ZIP -> H-4 features -> H-3 labels ->
reconcile label counts with the H-1 evidence -> validate against the feature
contract -> re-check "no future" on the real data -> write the files.
The original archives are only read, never modified.
"""
from __future__ import annotations

import argparse
import hashlib
import json
import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path(__file__).resolve().parent))
from sentinelai import features as F                        # noqa: E402
from sentinelai.contract import contract_dict, validate, write_contract   # noqa: E402
from sentinelai.labeling import RULE_VERSION, load_label_info   # noqa: E402


def md5(path: Path) -> str:
    h = hashlib.md5()
    with path.open("rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", default="./sentinelai_ait_data")
    ap.add_argument("--h1-dir", default="./h1_outputs")
    ap.add_argument("--network", default="fox")
    ap.add_argument("--out-dir", default="./h5_outputs")
    ap.add_argument("--future-checks", type=int, default=3)
    a = ap.parse_args()
    dl, h1, out = Path(a.data_dir) / "downloads", Path(a.h1_dir), Path(a.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    net, t_start = a.network, time.time()

    # 1. integrity: official MD5s as recorded by H-1 (from the Zenodo API)
    arch = pd.read_csv(h1 / "archive_checksums.csv", dtype=str, keep_default_na=False)
    official = dict(zip(arch["file"], arch["official_md5"]))
    zip_path = dl / f"{net}_netflows.zip"
    for p in (zip_path, dl / "label_info.txt"):
        got = md5(p)
        if got != official.get(p.name):
            sys.exit(f"[STOP] {p.name}: md5 {got} != official {official.get(p.name)}")
        print(f"[ok] {p.name} md5 {got} matches the official value")

    # 2-3. features + labels
    label_map = load_label_info(dl / "label_info.txt")
    flows = F.read_network_zip(zip_path, net)
    print(f"[read] {len(flows):,} flows from {zip_path.name}")
    labeled, unmapped = F.build_labeled_features(flows, label_map)
    print(f"[features] {len(labeled):,} labeled rows, {len(unmapped):,} unmapped rows left out")

    # 4. reconcile with H-1 evidence, per file
    q = pd.read_csv(h1 / "csv_file_quality_report.csv")
    q = q[q["network"] == net].set_index("file")
    recon = []
    for fname, g in pd.concat([labeled, unmapped]).groupby("source_file"):
        mine = {"rows": len(g), "normal_flows": int((g["y"] == 0).sum()),
                "suspicious_flows": int((g["y"] == 1).sum()), "unmapped_flows": int(g["y"].isna().sum())}
        h1row = {k: int(q.loc[fname, k]) for k in mine}
        recon.append({"file": fname, **{f"h5_{k}": v for k, v in mine.items()},
                      **{f"h1_{k}": v for k, v in h1row.items()}, "match": mine == h1row})
    recon = pd.DataFrame(recon)
    print(recon.to_string(index=False))
    if not recon["match"].all() or len(recon) != len(q):
        sys.exit("[STOP] label counts do not reconcile with H-1 evidence")

    # 5. contract
    problems = validate(labeled)
    if problems:
        sys.exit("[STOP] contract problems:\n  " + "\n  ".join(problems))
    print("[contract] feature table matches feature_contract_v0.json")

    # 6. no-future check on the REAL data: recompute on flows that ended before a cut
    full = F.build_features(flows).set_index("flow_id")
    rng = np.random.default_rng(42)
    checks = []
    for qtl in sorted(rng.uniform(0.1, 0.9, a.future_checks)):
        cut = float(flows["event_end_ms"].quantile(qtl))
        part = F.build_features(flows[flows["event_end_ms"] < cut]).set_index("flow_id")
        same = part[F.FEATURE_COLUMNS].equals(full.loc[part.index, F.FEATURE_COLUMNS])
        checks.append({"cut_quantile": round(float(qtl), 3), "cut_utc": str(pd.to_datetime(cut, unit="ms", utc=True)),
                       "rows_before_cut": len(part), "identical": bool(same)})
        print(f"[no-future] cut at {checks[-1]['cut_utc']}: {len(part):,} earlier rows identical={same}")
    if not all(c["identical"] for c in checks):
        sys.exit("[STOP] features changed when future flows were removed")

    # 7. write
    fpath = out / f"features_{net}.parquet"
    labeled.to_parquet(fpath, index=False)
    back = pd.read_parquet(fpath)
    assert validate(back) == [] and len(back) == len(labeled), "parquet round-trip failed"
    write_contract(out / "feature_contract_v0.json")
    unmapped[["flow_id", "source_file", "label_original"]].to_csv(out / f"unmapped_rows_{net}.csv", index=False)
    recon.to_csv(out / f"label_reconciliation_{net}.csv", index=False)
    labeled[F.FEATURE_COLUMNS].describe().T.to_csv(out / f"feature_stats_{net}.csv")

    sha = hashlib.sha256(fpath.read_bytes()).hexdigest()
    summary = {
        "network": net, "feature_file": fpath.name, "sha256": sha, "bytes": fpath.stat().st_size,
        "rows": len(labeled), "normal": int((labeled.y == 0).sum()), "suspicious": int((labeled.y == 1).sum()),
        "unmapped_rows_left_out": len(unmapped),
        "rows_by_file": labeled.groupby("source_file").size().to_dict(),
        "suspicious_by_file": labeled.groupby("source_file")["y"].sum().astype(int).to_dict(),
        "model_input_columns": F.FEATURE_COLUMNS,
        "contract_version": contract_dict()["contract_version"],
        "feature_module_version": F.FEATURE_MODULE_VERSION, "labeling_rule_version": RULE_VERSION,
        "time_range_utc": [str(pd.to_datetime(labeled.event_end_ms.min(), unit="ms", utc=True)),
                           str(pd.to_datetime(labeled.event_end_ms.max(), unit="ms", utc=True))],
        "no_future_checks": checks,
        "source_archive_md5": official[zip_path.name],
        "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
        "runtime_s": round(time.time() - t_start, 1),
    }
    (out / f"features_{net}.summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    print(json.dumps({k: summary[k] for k in ("feature_file", "rows", "normal", "suspicious",
                                              "unmapped_rows_left_out", "sha256", "runtime_s")}, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile sentinelai_h3h5/docs/H3_LABELING_RULE.md
# H-3 Labeling rule: AIT Netflow labels → normal (0) / suspicious (1)

**Rule version:** 1.0 · **Code:** `sentinelai/labeling.py` · **Tests:** `tests/test_labeling.py`
**Owner:** Habiba · **Checked by:** Malak (sign-off below)

## The rule

1. The only source of truth is the publisher's `label_info.txt` (Zenodo record 13168643,
   MD5 `e32c0e08d15725999df0c6a0c02aff47`; the code refuses any other copy).
2. **y = 0 (normal)** if the flow's label is listed under *Benign labels in &lt;its protocol&gt;*.
   **y = 1 (suspicious)** if it is listed under *Malicious labels in &lt;its protocol&gt;*.
3. Matching is **exact**: no trimming of spaces, no lower/upper-case changes, and the protocol must
   match. For example, `data exfiltration` is a UDP label, so it would not be mapped if it showed up in a TCP file.
4. Everything else is **unmapped (y = NA)**: an empty label, an unknown string, or a label listed only for the
   other protocol. Unmapped flows are **never treated as normal**. They are left out of training and
   evaluation labels, but they are still real traffic, so the feature module still uses them
   when it computes the 5-minute per-host windows.
5. The protocol comes from the file name: `tcp_complete.csv` and `tcp_nocomplete.csv` are TCP, and
   `udp_complete.csv` is UDP. H-1 checked this against each file's columns.

`label_info.txt` lists 13 benign and 39 malicious TCP labels, and 12 benign and 1 malicious UDP label.

## What this gives on the full dataset (H-1 evidence, 8 networks)

| | flows |
|---|---:|
| normal (0) | 2,845,522 |
| suspicious (1) | 618,878 |
| unmapped (left out) | 942 |
| **total** | **3,465,342** |

The 942 unmapped rows all come from `shaw/tcp_complete.csv`. Their label is empty. They are SMTP
connections (port 25) from two external hosts to the `collinsbecker_mail` server between 25 and 30 Jan 2022.
We do not guess a label for them.

## Points for Malak to look at

* **Labels the publisher calls benign even though an attacker host is involved**:
  `benign DNS - attacker requesting updates` (UDP) and `broken flow - benign` (TCP). We follow the publisher, so both are 0.
* **Half-open TCP is almost all attack traffic**: in `tcp_nocomplete.csv`, 390,756 of 394,636
  flows are suspicious. The feature `tcp_handshake_complete` will therefore be very strong. That is a real,
  observable signal (scans do not finish the handshake), but check how much of the model's skill comes from it alone.
* The role and network columns in the CSVs (e.g. `role_cli = attacker_0`) **give the answer
  away**, so they are not in the feature table. The IP addresses are kept as info columns (`src_ip`, `dst_ip`)
  for evidence, but they must never be model inputs: an IP stands for one machine, not for behaviour.

## Sign-off

- [ ] Malak: rule read, the tests pass (`python -m pytest -q tests/test_labeling.py`), and the counts above are accepted


In [ ]:
# H-3..H-5 Step 2: integrity check - every file written above must equal the copy that passed the tests outside Colab
import hashlib
TESTED = {"sentinelai/__init__.py": "d41d8cd98f00b204e9800998ecf8427e", "sentinelai/labeling.py": "45744bf443e08409cddb657e30bce88c", "sentinelai/features.py": "efe96bb929c651a835e526c31a67e3bf", "sentinelai/contract.py": "7775035019f2b958359a9348215b13df", "tests/real_headers.py": "f158afd1f84602ee96498d2c96e72476", "tests/conftest.py": "a1271fe469eeee7e5c89f3f9ac3d85cf", "tests/test_labeling.py": "6696f472988ee17ed6dc4fd6b50c62fd", "tests/test_features.py": "4f3137208f1ce9304ebd366526452c29", "h5_publish_features.py": "2cd6cc791c2d534e97c8f7eea83c0130", "docs/H3_LABELING_RULE.md": "4027d16071abb6659328224bf4be9749"}
ok = True
for f, want in TESTED.items():
    got = hashlib.md5(open(f'sentinelai_h3h5/{f}', 'rb').read()).hexdigest()
    ok &= got == want
    print(f'{str(got == want):5}  {got}  {f}')
assert ok, 'a file differs from the tested copy - do not continue'

In [ ]:
# H-3/H-4 Step 3: unit tests - H-3 rule + H-4 "windows never look into the future" (log kept in h5_outputs/pytest_log.txt)
!mkdir -p h5_outputs && cd sentinelai_h3h5 && python -m pytest -v -p no:cacheprovider 2>&1 | tee ../h5_outputs/pytest_log.txt | tail -40
last = open('h5_outputs/pytest_log.txt').read().strip().splitlines()[-1]
assert 'passed' in last and 'failed' not in last and 'error' not in last, last

In [ ]:
# H-5 Step 4: build + publish the first feature file (fox) with all checks; log kept in h5_outputs/run_log.txt
!python sentinelai_h3h5/h5_publish_features.py --data-dir ./sentinelai_ait_data --h1-dir ./h1_outputs --network fox --out-dir ./h5_outputs 2>&1 | tee h5_outputs/run_log.txt

In [ ]:
# H-5 Step 5: quick look at the published file (read back from disk, nothing recomputed)
import json
import pandas as pd
df = pd.read_parquet('h5_outputs/features_fox.parquet')
print(df.shape, '| y counts:', df['y'].value_counts().sort_index().to_dict())
print('\nFirst row, all columns:'); print(df.iloc[0].to_string())
print('\nMedian window features, normal (0) vs suspicious (1):')
print(df.groupby('y')[[c for c in df.columns if c.startswith('w5_')]].median().T.to_string())
print('\nRows by file and label:'); print(pd.crosstab(df['source_file'], df['y']).to_string())
print('\nFiles in h5_outputs:'); import os; [print(f'  {f:<34} {os.path.getsize("h5_outputs/"+f):>12,} bytes') for f in sorted(os.listdir('h5_outputs'))]

In [ ]:
# Bundle H-3..H-5 deliverables for download (Files panel -> h3_h5_outputs.zip): code, tests, docs, feature file, evidence
!rm -f h3_h5_outputs.zip && zip -r -q h3_h5_outputs.zip sentinelai_h3h5 h5_outputs -x "*/__pycache__/*" && ls -la h3_h5_outputs.zip && unzip -l h3_h5_outputs.zip

# Malak starts here (M-1, M-2, M-3)

**If the cell below says the file is missing** (Colab wiped its files): choose **Runtime → Run all**. That takes about 20–25 minutes and rebuilds everything from H-1 to H-5.

* `X` = the 28 input columns, `y` = the answer (0 normal, 1 suspicious). Never use `y`, `label_original`, `flow_id` or `event_end_ms` as inputs.
* Fox is for building and practising. Final tuning and test results wait for Habiba's H-6 split.
* Please read `sentinelai_h3h5/docs/H3_LABELING_RULE.md` (Files panel) and tell Habiba if you agree with the labeling rule.

In [ ]:
# Malak: load the cleaned data (fox network) and the list of the 28 model input columns
import json, pandas as pd
df = pd.read_parquet('h5_outputs/features_fox.parquet')
X_cols = json.load(open('h5_outputs/feature_contract_v0.json'))['model_input_columns']
X, y = df[X_cols], df['y']
print(X.shape, '| normal:', int((y == 0).sum()), '| suspicious:', int((y == 1).sum()))
X.head()